# Max's Psychedelics Project Master QC Script
## (after only allowing longitudinal participants)

## HOW TO USE THIS NOTEBOOK:
### Chat with Max BEFORE using the first time
1. Do not begin unless you are confident you have an hour to do everything (in our busiest period). 
2. Make sure you are connected to Yale VPN and have opened smb://storage.yale.edu/home/Powers_Lab-CC1095-Psychiatry/Max/psychedelics in your file explorer/finder (To connect see the Powers Lab Manual instructions for connectin to Powers Lab Network Drive)
3. Run the "Load libraries" cell
4. (If you are new to the project) Add your full name with capitalization etc. to the elif statements in the "ID yourself..." cell. 
5. Run "Run QC" Cell. You will need to copy/paste IP addresses from the PDF Archive on REDCap's file repository.
6. Do whatever it tells you TODAY (please!). If for some reason you cannot get through everything, PLEASE let Max or someobody else know so they can finish. If it pauses for a long time, it is waiting for a typed input from you -- this will be at the top in VS code and will tell you what to input. Just make sure you go through the cells in order (some days you only need to do the NEW RECORDS or the COMPLETED RECORDS scections. Probably usually both). You will need to manually enter nubmers into spydialer and tell the script if they are voips or not. You will have nice, neat lists of records with emails and relevant email templates typed out in a directory named with today's date. 

NOTE!!!! If this is your first time, then please make sure you have the relevant packages installed by running the following commands from terminal:
pip install pandas
pip install numpy
pip install statsmodels
pip install matplotlib
pip install seaborn
pip install scikit-learn
pip install jsonlib
pip install gzip
pip install base64
pip install csv
pip install scipy
pip install pingouin
pip install datetime

You will need to have installed python and jupyter notebooks first :) 

# Load Libraries

In [1]:
#Import libraries
import pandas as pd
import numpy as np


import json
import gzip
import base64
import csv
from io import BytesIO
import scipy.io
from scipy.io import loadmat, savemat
from scipy.io import loadmat, savemat
from IPython.display import HTML, display
import re


from math import sqrt
import pingouin as pg

from datetime import datetime, timedelta

import re
import os
import glob

from redcap import Project

import statsmodels.api as sm
import statsmodels.formula.api as smf


#Stop pandas from downcasting during replace operations to silence the warning messages
pd.set_option('future.no_silent_downcasting', True)

# ID yourself and then pull REDCap Records that need to be checked

In [2]:
############ ID User and Load Data #####################
### Places to replace names (the only thing you need to change!)
########################################################

user = input("Who is using the script? m, Kayla, or Gabby?")
if user.lower() == "m":
    myname = "Maximillian S. Greenwald"
elif user.lower() == "kayla":
    myname = "Kayla Morgan"
elif user.lower() == "gabby":
    myname = "Gabriela Hernandez"
else:
    print(f"Unrecognized user: {user}. Allowed users = Max, Kayla, and Gabby.\n If you are a new user, please add an elif statement to the top of this cell with your full name with proper capitalization.")
    exit()

sharedrive_network_path = "/Volumes/psychedelics/online/"
ip_path = sharedrive_network_path + "ips/ips_full.csv"
json_path = sharedrive_network_path + "jsons/failed_task_jsons_baseline.csv"

########################################################
########################################################
########################################################

### FIRST check to make sure that there are no incomplete flags from the most recent QC run:
# Check for existing INCOMPLETE flags in the most recent date_directory

def check_incomplete_flags(base_path):
    try:
        date_dirs = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))], reverse=True)
        date_dirs = [d for d in date_dirs if d.startswith('2')]  # only select directories that start with 2 and therefore is a date in YYYYMMDD format
    except FileNotFoundError:
        print("!!! You have to actually open the Powers Lab network drive directly to /psychedelics/online/qc_to_dos !!!")
        exit()
    if date_dirs:
        latest_dir = os.path.join(base_path, date_dirs[0])
        print(f"Script last run on {date_dirs[0]}. Checking for INCOMPLETEs...")
        incomplete_flags = [f for f in os.listdir(latest_dir) if "INCOMPLETE" in f]
        if incomplete_flags:
            for flag in incomplete_flags:
                flag_path = os.path.join(latest_dir, flag)
                try:
                    with open(flag_path, 'r') as file:
                        baduser = file.read().strip()
                except:
                    baduser = "unknown user"
                
                print(f"INCOMPLETE flags detected in {latest_dir} completed by {baduser}!!! Please email {baduser} or Max and ask them to address the following:")
                print(f"- {flag}")
            print("Do NOT run QC until these are resolved!")
            exit()
    else:
        print("No previous date directories found. Proceeding with QC.")

BASE_PATH = f"{sharedrive_network_path}qc_to_dos"

check_incomplete_flags(BASE_PATH)

### Read in data
baseline_api = "1D481003114ECDA8E4077078E0D08D0A"
redcap_api = "https://redcap.research.yale.edu/api/"
project = Project(redcap_api, baseline_api)
# Export data and get record_id as a column
df = project.export_records(format_type='df')
df = df.reset_index(names='record_id')



rpt_api = "66785AFE5341D3F73B4F518339C60186"



#remove the test records
testingrecords = ['BlueLightTesting_1','r/Drugs Testing','TESTING','TESTING2','TESTING3','TESTING4','TESTING5','TESTING6','test_rec_w_realdata_ithink','testing_before_newqc','testing_max_prerelaunch','testing_pre_yale_relaunch']
df = df[~(df['record_id'].isin(testingrecords))]
df.fillna({'replay_email': "NO EMAIL GIVEN"},inplace=True)


#convert record id and student to integer
df['record_id']=df['record_id'].astype(int)
df['student_yn']=df['student_yn'].fillna(0)
df['student_yn']=df['student_yn'].astype(int)

df_og = df.copy()

### Check to see whether or not we need to close the study!!
confirmed_participants = len(df_og[df_og['qc_passed']>0])
confirmed_nonyale_participants = len(df_og[(df_og['qc_passed']>0) & (df_og['student_yn']<1)])
# Calculate people who have been cleared to start AND have not exceeded 3 days
df_recent = df_og[df_og["record_id"] > 900].copy()  # Filter for records with record_id > 900
df_recent["today_survey_bl"] = pd.to_datetime(df_recent["today_survey_bl"], format="%Y-%m-%d")# First convert timestamp_survey_bl to datetime if it's in  format="%Y-%m-%d %H:%M:%S":
df_recent["days_since_survey"] = (datetime.now() - df_recent["today_survey_bl"]).dt.days
enrolled_participants = len(df_recent[(df_recent['eligible_click']>0) & (df_recent['qc_passed'].isna()) & (df_recent["days_since_survey"]<3) & (df_recent['exceed_3_vch']<3) & (df_recent['exceed_3_ach']<3) & (df_recent['exceed_3_prl']<3)])#& ~(df_og['exceed_3_spacejunk']>0)])
if confirmed_nonyale_participants + enrolled_participants > 310:
    print(f"***\n, IMPORTANT!!!!!!! Please email or slack Max to let him know that we have {confirmed_nonyale_participants} confirmed non-yale participants and {enrolled_participants} currently enrolled participants. We are at {confirmed_participants} total participants, which is more than 310. We need to close the study to new participants.\n****")
    def UpdateRedcap(records,fields,data,proj,newval=None,overwrite=None,convert_to_int=False):

        # Create a new dataframe with the records and fields
        update_df = data[data['record_id'].isin(records)]

        if newval:
            for field in fields:
                update_df[field] = newval
        
        allfields = fields + ['record_id']
        update_df = update_df[allfields]
        if convert_to_int==True:
            for field in allfields:
                update_df[field] = update_df[field].astype('Int64')
        if overwrite:
            proj.import_records(to_import=update_df,import_format='df',overwrite=overwrite)
        else:
            proj.import_records(to_import=update_df,import_format='df')

    # update the field in REDCap so max gets an email
    UpdateRedcap(records=[200],fields=['close_study'],data=df_og,proj=project,newval="close",overwrite=None,convert_to_int=False)

### Check to see whether we have maxed out our control subjects!!
confirmed_control_participants = len(df_og[(df_og['qc_passed']>0) & (df_og['control_group']>0) & (df_og['psycheduse_yn']>1)])
if confirmed_control_participants >= 20:
    print(f"***\n, IMPORTANT!!!!!!! Please email or slack Max to let him know that we have {confirmed_control_participants} confirmed control participants. We need to close the study to new control participants.\n****")
    def UpdateRedcap(records,fields,data,proj,newval=None,overwrite=None,convert_to_int=False):

        # Create a new dataframe with the records and fields
        update_df = data[data['record_id'].isin(records)]

        if newval:
            for field in fields:
                update_df[field] = newval
        
        allfields = fields + ['record_id']
        update_df = update_df[allfields]
        if convert_to_int==True:
            for field in allfields:
                update_df[field] = update_df[field].astype('Int64')
        if overwrite:
            proj.import_records(to_import=update_df,import_format='df',overwrite=overwrite)
        else:
            proj.import_records(to_import=update_df,import_format='df')

    # update the field in REDCap so max gets an email
    UpdateRedcap(records=[200],fields=['close_control'],data=df_og,proj=project,newval="close",overwrite=None,convert_to_int=False)

#Remove the old yale ids
df = df[df['record_id']>202]

df = df[~(df['record_id']==456)]
df = df[~(df['record_id']==654)]


#Remove the records for people who didn't move past the consent
df = df[~(df['psych_spectrum_v2'].isna())]



#### Get the current instrument names (since we may end up changing records data)
task_instruments = {}
for task in ['auditory_ch_task','prl_task','visual_ch_task','spacejunk_game']:
    try:
        metadata = project.export_metadata(format_type='df')
        task_instruments[task] = [x for x in metadata[metadata['form_name'] == task].reset_index(names='field_name')['field_name'].tolist() if 'replay' not in x and x in df.columns]
    except Exception as e:
        print(f"Not happening for {task}: {e}")


########### SPECIFY RECORDS TO CHECK

### GROUP 1: Records that have finished/have all task data but not had their QC completed

newrecords = df[(~(df['race_qc'].isna())) & (df['qc_passed'].isna() & (df['screening_pass']>0))]
for taskdata in ['task_data_ach_task_short_baseline','task_data_prltask','task_data_vch_short_psychedelic_bl','task_data_spacejunk_bl']:
    newrecords = newrecords[~(newrecords[taskdata].isna())]

# Remove people who have been flagged for inconsistent SP answers ("verify_emailed" is 1) but have not filled out the form ("sp_tot_verify" is NA") or have not been checked by Max ("sp_verify_pass" is NA):
newrecords = newrecords[~((newrecords['verify_emailed']==1) & (newrecords['sp_tot_verify'].isna()))]
newrecords = newrecords[~((newrecords['verify_emailed']==1) & (newrecords['sp_verify_pass'].isna()))]


# Add yale students who have finished the PRL task but not done SMS verification this list
df_raw_yale = df[~(df['task_data_prltask'].isna()) & (df['student_yn']==1) & (df['qc_passed'].isna() & (df['record_id']>1000))] # only get NEW yale students to avoid accidentally emailing old ones hehe



# If there are new records, add them to the list of records to check -- create 2 lists for yale and regular participants. 
if len(newrecords) > 0:
    rec_to_check_reg = newrecords['record_id'].tolist()
else:
    rec_to_check_reg = []
if len(df_raw_yale) > 0:
    rec_to_check_yale = df_raw_yale['record_id'].tolist()
else:
    rec_to_check_yale = []
records_to_check = rec_to_check_reg + rec_to_check_yale

records_to_check = list(set(records_to_check))  # Remove duplicates

if len(records_to_check) <1:
    print("No new study completions since script was last run")
else:
    print(f"{len(records_to_check)} new records to check!!!!")


### GROUP 2: Records that have been found eligibile and need to be checked (+ people who do the longitudinal waiting and/or say they would do a zoom call to discuss cannabis washout)
eligible = ()
records_to_screen = df[~(df["submit_screen_v3"].isna()) & (df['screening_pass'].isna()) & (df['qc_passed'].isna()) & (~(df['phone_number'].isna())) & (df["ip_zoom_invite"].isna())]['record_id'].tolist()

# Add people who do the longitudinal waiting but haven't been screened
longitudinal_screens = df[~(df['sp_wait_regret'].isna()) & (df['screening_pass'].isna()) & (df['qc_passed'].isna() & ~(df['phone_number'].isna()) & (df["waiting_emailed_yn"].isna()))]['record_id'].tolist()
records_to_screen = records_to_screen + longitudinal_screens

# Add people who say they would do a zoom call to discuss cannabis washout (and haven't been screened)
cannabis_screens = df[~(df['zoom_email'].isna()) & (df['screening_pass'].isna()) & (df["cb_email_yn"].isna())& ~(df['phone_number'].isna())]['record_id'].tolist()
records_to_screen = records_to_screen + cannabis_screens


records_to_screen = list(set(records_to_screen))  # Remove duplicates


if len(records_to_screen) <1:
    print("No new records to screen since script was last run")
else:
    print(f"{len(records_to_screen)} new records to screen!!!!")

### Internal flags to ensure folks do this in the correct order
qc_run = False
spydialed = False
screen_todos_made = False
screens_updated = False
recs_updated = False
recs_todos_made = False

### Check for people who have newly finished the addon study
addon_finishes = df[(df["addons_completed"].isna()) & (df["trauma_addon_yn"].notna())]['record_id'].tolist()
if len(addon_finishes) > 0:
    print(f"{len(addon_finishes)} new folks finished the brief add-on questions so just need to be payed -- run the RUN QC cell and the ensuing cells")

if (len(records_to_screen) <1) & (len(records_to_check) <1) & (len(addon_finishes) <1):
    print("Nothing to do today! DONT DO ANYTHING ELSE!")

Script last run on 2026-02-24. Checking for INCOMPLETEs...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/redcap/methods/base.py:177: DtypeWarning: Columns (0,15,16,19,21,28,29,30,32,51,53,61,63,64,68,69,70,72,77,78,79,80,82,125,204,256,259,350,383,406,430,431,435,450,456,458,462,510,517,539,545,548,551,553,554,564,578,590,595,601,604,607,608,610,612,614,627,642,672,674,834,852,859,876,888,889,896,897,904,905,912,913,920,921,1210,1357,1359,1594,1743,1752,1764,1767,1775,1779,1791,1800,1806,1810,1811,1818,1822,1873,1874,1875,1876,1877,1878,1879,1880,1881,1894,1898,1901,1903,1912,1913,1924,1932,1933,1935,1937,1938,1939,1976,1977,2119,2120,2123,2124,2127,2128,2129,2135,2137,2138,2142,2143,2145,2146,2147,2150,2151,2174,2175,2178,2182,2183,2186,2504,2531,2533) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(buf, **df_kwargs)


No new study completions since script was last run
1 new records to screen!!!!


<span style="font-size: 80px;"> RUN QC </span>

## [Click me to copy and paste IP addresses!](https://redcap.research.yale.edu/redcap_v15.5.36/index.php?pid=604&route=FileRepositoryController:index&type=pdf_archive)

In [3]:
### First, for all the participants who need to be screened

#reset the index and make a copy so it isn't fragmented and gets mad
df = df.reset_index()
df = df.copy()

####################################################################################
############ IP ADDRESS SCREENING FOR NEW RECORDS TO SCREEN ########################
####################################################################################
### If we have new records to screen, then we need to get the IP addresses from the REDCap file repository and do a fraud assessment
if len(records_to_screen) > 0:
    ### Known orgs and countries associated with fraud
    forbidden_orgs = ["AS14061 DigitalOcean, LLC","AS396356 Latitude.sh",
                     "AS5650 Frontier Communications of America, Inc.",
                     "AS25769 Garden Valley Telephone Company","AS21859 Zenlayer Inc",
                      "AS22200 Bloomingdale Communications Inc.",#"AS63023 GTHost",
                     "AS8167 V tal", "AS11878 tzulo, inc.",
                     "AS393398 1515 ROUNDTABLE DR PROPERTY, LLC",
                     'AS396956 Meriwether Lewis Electric Cooperative', 'AS9009 M247 Europe SRL', 'AS16276 OVH SAS', 
                     'AS136787 PacketHub S.A.', 'AS62240 Clouvider', 'AS60068 Datacamp Limited',
                     'AS6079 RCN', 'AS13285 TalkTalk Communications Limited',
                     'AS6461 Zayo Bandwidth', 'AS174 Cogent Communications', 'AS212238 Datacamp Limited', 
                     'AS6300 Consolidated Communications, Inc.'
                     ]

    forbidden_country_names = ["Nigeria","Ghana"]  
    # Display a hyperlink to the REDCap file repository
    display(HTML(f'<a href="https://redcap.research.yale.edu/redcap_v15.0.20/index.php?pid=604&route=FileRepositoryController:index&type=pdf_archive" target="_blank">CLICK ME AND COPY ROWS ABOVE {min(records_to_screen)}</a>'))
    # Prompt the user for input
    grossstring = input("1) Go to the link BELOW the cell you just ran (PDF Snapshot Archive in the File Repository)\n" + 
                        "2) Highlight all rows from the top down to Record " + f"{min(records_to_screen)}\n" + 
                        "3) Copy and paste ALL of it here!\n" + 
                        "Type 'no ip' if you have Max's OK to continue without IP addresses.\n" +
                        "Type 'stop' if you ran this cell accidentally and want to abort.")

    # Handle special inputs
    if grossstring.lower() == 'stop':
        print("Process aborted by user.")
        raise Exception("Process intentionally stopped by user")
    
    if grossstring.lower() == 'no ip':
        print("Continuing without IP address verification (with Max's approval).")
        new_ips = pd.DataFrame(columns=['record_id', 'ip'])
    else:
        # Parse the input string to extract record_ids and IP addresses
        def parse_redcap_log(log_string):
            # Find all record IDs and IPs in the entire text
            record_matches = re.findall(r"(\d+)\s*\tConsent", log_string)
            ip_matches = re.findall(r"\d{2}-\d{2}-\d{4}\s+\d{2}:\d{2}\s+(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s*\t1\t e-Consent", log_string)
            
            # Convert record IDs to integers
            record_ids = [int(id) for id in record_matches]
            
            # Create entries with matching indices
            entries = []
            for i in range(min(len(record_ids), len(ip_matches))):
                entries.append({
                    'record_id': record_ids[i],
                    'ip': ip_matches[i]
                })
            
            # Create DataFrame
            ip_data = pd.DataFrame(entries)
            
            # Check if there are any issues with the parsing
            if len(record_ids) != len(ip_matches):
                return None, f"Warning: Found {len(record_ids)} record IDs but {len(ip_matches)} IP addresses. You are probably not copying the entirety of a row or two -- zoom out on the REDCap page and make sure you go from the size column to the name column."
            
            if ip_data.empty:
                return None, "No matching entries found. Please check the format of the copied text and try again."
            
            return ip_data, None

        # Process the input with a retry loop
        max_attempts = 3
        attempt = 0
        new_ips = None
        
        while attempt < max_attempts and new_ips is None:
            if attempt > 0:
                print(f"Attempt {attempt+1}/{max_attempts}. Please try again.")
                grossstring = input("INVALID INPUT -- NO IPS DETECTED! Copy and paste ALL rows from the PDF Snapshot Archive in the File Repository:\n" +
                                  "Type 'no ip' if you have Max's OK to continue without IP addresses.\n" +
                                  "Type 'stop' if you want to abort.")
                
                if grossstring.lower() == 'stop':
                    print("Process aborted by user.")
                    raise Exception("Process intentionally stopped by user")
                
                if grossstring.lower() == 'no ip':
                    print("Continuing without IP address verification (with Max's approval).")
                    new_ips = pd.DataFrame(columns=['record_id', 'ip'])
                    break
            
            new_ips, error_message = parse_redcap_log(grossstring)
            
            if error_message:
                print(error_message)
                attempt += 1
        
        if new_ips is None:
            print("Maximum retry attempts reached. Moving on with empty IP data. This may cause issues with the IP screening process.")
            new_ips = pd.DataFrame(columns=['record_id', 'ip'])
        # else:
            # print(f"Successfully extracted {len(new_ips)} record/IP pairs.")

    #### Now get additional IP information for each IP and, if we haven't already done this for this record, add it to the ips database
    # Load the existing IP database
    df_ips = pd.read_csv(ip_path)
    df_ips = df_ips.drop(columns=[x for x in df_ips.columns if 'dumb' in x])
    df_ips['record_id'] = df_ips['record_id'].astype(int)


    # Trim new IPs to only include records that are not already in the database
    if not new_ips.empty:
        new_ips['record_id'] = new_ips['record_id'].astype(int)
        new_ips = new_ips[~(new_ips['record_id'].isin(df_ips['record_id']))]

    if not new_ips.empty:
        ### Set up to analyze the data
        import ipinfo
        access_token = "20f981656f0139"
        handler = ipinfo.getHandler(access_token)

        ### iterate through dataframe and get details for each ip
        ipdeets = []
        for index,row in new_ips.iterrows():
            details = handler.getDetails(row['ip'])
            newipdeets = pd.DataFrame(details.all,index=[0]) #other stuff accessible as attributes -- eg details.loc gives latitude, longitude, etc. details.hostname gives hostname etc.
            newipdeets['record_id'] = row['record_id']
            ipdeets.append(newipdeets)

        ipstufftoadd = pd.concat(ipdeets,ignore_index=True)
        df_ips_new = pd.concat([df_ips,ipstufftoadd])

        # Save the new df_ips to a CSV file -- make sure we haven't messed up the merge or whatever
        if (len(df_ips_new) - len(df_ips)) == len(ipstufftoadd):
            # print("New IPs saved to the database.")
            df_ips_new.to_csv(ip_path, index=False)
            df_ips = df_ips_new.copy()

    else:
        print(f"Records to screen for today have all already been screened before (we already knew their IP address)!!!\n Please double check in REDCap to make sure that the following records are actually at the point where they need to be screened: {records_to_screen}")

    # If any records match a forbidden org, country_name, etc. then add them to DontPassGo list so that we know not to pay them. 
    DontPassGo = [] # People using known fraudulent info or very obviously using a repeat email to double dip or get around eligibility requirements
    for record in records_to_screen:
        if any(df_ips.loc[df_ips['record_id']==record,'org'].isin(forbidden_orgs)) or any(df_ips.loc[df_ips['record_id']==record,'country_name'].isin(forbidden_country_names)):
            # print(f"Record {record} has an ip address org or country name known to be associated wtih fraud")
            DontPassGo.append(record)

    # If any records 


### Check eligibility to find eligibility dodgers, and also remove anybody failing some basic challenge questions
if len(records_to_screen)>0: # Create a new column for eligibility that's an empty string
    df_screens = df[df['record_id'].isin(records_to_screen)].copy()
    df_screens["ineligibilty_reason"] = ""

    for index, row in df_screens[~(df_screens["qc_passed"]==1) & (df_screens["screening_survey_complete"]>0)].iterrows():
        # if row[columns_to_check].isna().any():
        #     df_screens.loc[index, 'eligible'] = np.nan
        if row['age_v2'] > 65:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'Old '
        if row['age_v2'] < 18:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'Young '

        if row['cognition_screener_v2'] > 0:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'Neurocog Issues '

        if row['seizure_hx_v2'] > 0:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'Seizure History '

        if row['intox_screen_v2'] > 0:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'High During Intake '

        if row['psycheduse_yn'] > 1:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'No SP Use '

        if row['raven_total_score_v2'] < 1:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index, 'ineligibilty_reason'] += 'Low Raven '
        # if (
        #         (row['activecannabisuse_lastuse'] < 28 and row['cannabis_frequency'] > 9) or
        #         (row['activecannabisuse_lastuse'] < 14 and row['cannabis_frequency'] in [8, 9]) or
        #         (row['activecannabisuse_lastuse'] < 7 and row['cannabis_frequency'] == 7) or
        #         row['activecannabisuse_lastuse'] < 3
        # ):
        #     df_screens.loc[index, 'eligible'] = 0
        #     df_screens.loc[index,'ineligibilty_reason'] += 'CB Use Too Recent '
        if row['no_computer'] > 0:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index,'ineligibilty_reason'] += 'No Computer '
        # if row['atypical_since_sp'] > 0:
        #     df_screens.loc[index, 'eligible'] = 0
        #     df_screens.loc[index,'ineligibilty_reason'] += 'Atypical More Recent '
        # if row['sp_naiive'] < 1:
        #     df_screens.loc[index, 'eligible'] = 0
        #     df_screens.loc[index,'ineligibilty_reason'] += 'No SP Use '
        if row['english_fluency'] <1:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index,'ineligibilty_reason'] += 'No English '
        if pd.isna(row["geo_crit"]):
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index,'ineligibilty_reason'] += 'Fraudster Region '
        if row["screening_pass"] < 1:
            df_screens.loc[index, 'eligible'] = 0
            df_screens.loc[index,'ineligibilty_reason'] += 'Fraudster Phone or IP '
        # else:
        #     df_screens.loc[index, 'eligible'] = 1
        #     df_screens.loc[index,'ineligibilty_reason'] += 'Eligibile'

        ### Check challenge questions
        # If they say the haven't used SPs adn then that they have
        if (row["control_group"]>0) and (int(row['psycheduse_yn'])!=2):
            DontPassGo.append(row['record_id'])
        
        # If they learned about the study through yale intro psych and aren't a student
        if (row['student_yn']>0) and (row['howtheyfoundus']<2):
            DontPassGo.append(row['record_id'])


    # Remove trailing whitespace from ineligibilty_reason
    df_screens['ineligibilty_reason'] = df_screens['ineligibilty_reason'].str.strip()

    



####################################################################################
############ ELIGIBILTY DODGER (IP/Email) CHECK -> SPYDIALER LIST ##################
####################################################################################
## Make sure they didn't just do the whole screening, get found as ineligible, and then repeat OR have already done the whole thing, gotten paid, or even gotten marked as bad and are repeating. 
# Get a list of people who DID finish the screening survey but DIDNT get the submit screen option... 
if len(records_to_screen) > 0:
    ineligiblemask = (~(df["datedone_screening_survey"].isna()) & (df['submit_screen_v3'].isna()))
    finishedalready = ~(df["qc_passed"].isna())
    ineligibles = df[ineligiblemask|finishedalready]
    ineligibileemails = ineligibles['payment_email_bl'].tolist() + ineligibles['email_addtl_contact'].tolist() + ineligibles['interested_cntrlstudy'].tolist() + ineligibles["interested_spstudy"].tolist()
    ineligibileemails = [email for email in ineligibileemails if pd.notna(email)] # have to remove NaNs otherwise a bunch of people will match hehe
    # Also get the IP addresses of the ineligible records -- excluding for known safe orgs that are likely to have repeat ip addresses
    safe_orgs = ["AS14550 Middlebury College","AS29 Yale University"]
    trueips = df_ips[~(df_ips['org'].isin(safe_orgs))]
    ineligibile_ips = trueips[trueips['record_id'].isin(ineligibles['record_id'])]['ip'].tolist()


    # Lists we will generate 
    sus_ips = [] # People who have the same IP address as someone who has already been marked as ineligible or completed the study -- require Zoom followup 
    spydialer_check = [] # People who do not outright fail and just need to have their number checked in spydialer 
    email_max = [] # People who RAs should email me to ask about

    for record in [x for x in records_to_screen if x not in DontPassGo]:
        thisrecdf = df.loc[df['record_id']==record]
        thisguysemails = thisrecdf['payment_email_bl'].tolist() + thisrecdf['email_addtl_contact'].tolist() + thisrecdf['interested_cntrlstudy'].tolist() + thisrecdf["interested_spstudy"].tolist()
        
        # Check if the record exists in df_ips
        if record in df_ips['record_id'].values:
            thisip = df_ips.loc[df_ips['record_id']==record,'ip']
        else:
            # Handle case where record is not in df_ips
            thisip = pd.Series([], dtype='object')  # Empty series that won't trigger .isin() condition
            email_max.append(record)  # Add to email_max list for follow-up
            print(f"Record {record} doesn't have an IP address? - please email Max for follow-up")

        # If the record has used an email in an ineligible or already completed record (and it isnt this same email lol), flag and mark as ineligible. 
        for email in thisguysemails:
            if email in ineligibileemails:
                # Find which ineligible record has this email
                matching_ineligible = ineligibles[
                    (ineligibles['payment_email_bl'] == email) |
                    (ineligibles['email_addtl_contact'] == email) |
                    (ineligibles['interested_cntrlstudy'] == email) |
                    (ineligibles['interested_spstudy'] == email)
                ]

                # If this isn't just the record we are currently checking, mark as ineligible
                if record not in matching_ineligible['record_id'].tolist():
                    DontPassGo.append(record)
                    print(f"Record {record} has email ***{email}*** which matches with ineligible and/or completed record(s) {matching_ineligible['record_id'].tolist()}")
                    print(f"Behold:\nFraudster info:\n{matching_ineligible[['record_id','qc_passed','datedone_consent_baseline','datedone_screening_survey','payment_email_bl','email_addtl_contact','interested_cntrlstudy','interested_spstudy']]}\n{record} info:\n{thisrecdf[['record_id','qc_passed','datedone_consent_baseline','datedone_screening_survey','payment_email_bl','email_addtl_contact','interested_cntrlstudy','interested_spstudy']]}")
                    break  # No need to check other emails once we find a match
                else:
                    pass

        # If they aren't an eligibility dodger... 
        if record not in DontPassGo:
        # Check if the IP address is in the list of ineligible IPs and then have them check the spydialer... 
            # Check if the IP address is in the list of ineligible IPs
            if not thisip.empty and thisip.isin(ineligibile_ips).any():
                # Get matching records with this IP
                matching_records = ineligibles[ineligibles['record_id'].isin(trueips[trueips['ip'].isin(thisip.values)]['record_id'].tolist())]['record_id'].tolist()
                
                # Only flag as suspicious if the matching records aren't just this same record
                if record not in matching_records or len(matching_records) > 1:
                    # Get the organization info safely
                    org_info = df_ips.loc[df_ips['ip'].isin(thisip.values), 'org'].iloc[0] if not df_ips[df_ips['ip'].isin(thisip.values)].empty else "Unknown"
                    
                    # Filter out the current record from the matching records list for display
                    other_matching_records = [r for r in matching_records if r != record]
                    
                    print(f"Record {record} has IP address ({thisip.values[0]}; org = {org_info}) that is IDENTICAL to the following record(s) that were found ineligibile/completed the study already:\n{other_matching_records}")
                    sus_ips.append(record)

            spydialer_check.append(record)

# # If we have any records requiring spy dialer check, let the user know!
# if len(spydialer_check) > 0:
#     print(f"{len(spydialer_check)} records appear eligibile but need to be checked with spydialer!\nGo to the NEW RECORDS SCREENING cells and go through them one by one")

### If we have records to screen, check that every record gets categorized into one of the above lists -- if not flag it. If we get through all records, give the user the messsage to run all the cells in order going down from NEW RECORDS SCREENING
if len(records_to_screen) > 0:
    for record in records_to_screen:
        if record not in DontPassGo and record not in spydialer_check and record not in sus_ips:
            print(f"Screening Record {record} couldnt be categorized!! Message Max to check the data for this record.")
            # raise Exception(f"Record {record} was not categorized into any of the lists. Please check the data for this record.")
    print(f"***\n{len(spydialer_check)} Screening records to check the numbers for and at least {len(DontPassGo)} who need to be rejected!\nPlease run all cells in order going down from NEW RECORDS SCREENING\n****")


### Limit data to those we want (and make sure they have all the task data) to check and reset the index
df_raw = df[df['record_id'].isin(records_to_check)]

skippedscreen = df_raw[(df_raw["screening_pass"].isna())|(df_raw["screening_pass"] == 0)]['record_id'].tolist()

# If any control subjects report psychedelic use since screening flag them
control_notes = df[(df["control_group"]>0) & (df["halluc_sincescreen"]<2)]['record_id'].tolist()

##Used to exclude folks w/o the task data but theoretically they shouldnt be doing sms verification w/o all of their data, and yale students wont do all of them
no_prl = []
no_ach = []
no_vch = []
no_spacejunk = []

for index,row in df_raw.iterrows():
    if pd.isna(row['task_data_ach_task_short_baseline']) and pd.isna(row['task_data_ach_bl_retrieved']):
        no_ach.append(row['record_id']) 
    if row['student_yn']==0:
        if pd.isna(row['task_data_prltask']) and pd.isna(row['task_data_prl_bl_retrieved']):
            no_prl.append(row['record_id'])
        if pd.isna(row['task_data_vch_short_psychedelic_bl']) and pd.isna(row['task_data_vch_bl_retrieved']):
            no_vch.append(row['record_id']) 
    else:
        if pd.isna(row['task_data_spacejunk_bl']) and pd.isna(row['task_data_spacejunk_bl_retrieved']):
            no_spacejunk.append(row['record_id'])
records_to_check = df_raw['record_id'].tolist()

df_raw = df_raw.reset_index(drop=True)


### Fucking fraudsters who are pasting random strings in the retrieved boxes need to be marked as fucking scammers
fraud_copy_paste_ach = []
fraud_copy_paste_vch = []
fraud_copy_paste_prl = []
fraud_copy_paste_spacejunk = []

# # For now, if they have both missing data and a retrieved string, assume they're a scammer and remove from the whole dataframe to reduce risk of error
# for task,retrievedtask,fail_list in zip(["task_data_ach_task_short_baseline","task_data_prltask","task_data_vch_short_psychedelic_bl","task_data_spacejunk_bl"],["task_data_ach_bl_retrieved","task_data_prl_bl_retrieved","task_data_vch_bl_retrieved","task_data_spacejunk_bl_retrieved"],[fraud_copy_paste_ach,fraud_copy_paste_vch,fraud_copy_paste_prl,fraud_copy_paste_spacejunk]):
#     scammer_mask = ((df_raw[task].isna()) & (df_raw[retrievedtask].notna()))
#     likelyscammers = df_raw[scammer_mask][['record_id','email_addtl_contact',task,retrievedtask]]
#     likelyscammers.to_csv(f"/Volumes/psychedelics/online/scamdfs/{task}_scammers_{datetime.now().strftime('%Y-%m-%d')}.csv")
#     fail_list.extend(likelyscammers['record_id'].tolist())
#     print(f"{len(fail_list)} scammers detected for {task} -- check the CSV file to confirm no good faith pasted strings")

#     # remove them!
#     df_raw = df_raw[~(df_raw['record_id'].isin(fail_list))]

# Set all of the people who have not passed QC and have weird copy-pasted strings for the retrieved fields, mark these as nan
for retrievedtask in ["task_data_ach_bl_retrieved","task_data_prl_bl_retrieved","task_data_vch_bl_retrieved","task_data_spacejunk_bl_retrieved"]:
    df.loc[df['qc_passed']<1,retrievedtask] = np.nan
    df_raw.loc[df_raw['qc_passed']<1,retrievedtask] = np.nan    

df_raw = df_raw.reset_index(drop=True)

# For doing the task analysis do not include the fruadster strings cuz it results in errors
scammers = fraud_copy_paste_ach + fraud_copy_paste_vch + fraud_copy_paste_prl + fraud_copy_paste_spacejunk
df_nocopypaste = df[~(df['record_id'].isin(scammers))].copy()

#Calculate phq9 score teehee
df_raw['phq9_tot'] = df_raw[['phq1', 'phq2', 'phq3', 'phq4', 'phq5', 'phq6', 'phq7', 'phq8', 'phq9']].sum(axis=1)
df_raw_yale['phq9_tot'] = df_raw_yale[['phq1', 'phq2', 'phq3', 'phq4', 'phq5', 'phq6', 'phq7', 'phq8', 'phq9']].sum(axis=1)

#############################
### Attention Checks and Challenge questions####
#############################
###Function that takes a list of columns for a questionnaire or something and adds the total for each row and then appends a column w/the total score
def GetTotalScore(df,listOfColumnsToAddTogether,TotalScoreColumnName):
    df[TotalScoreColumnName] = df.apply(lambda row: row[listOfColumnsToAddTogether].sum(),axis=1)
### Assess who is failing the attn check
# Get a list of participants who did not pass the attn_check
attnchecks = ['attn_check_surveybl', 'attn_check_surveybl2', 'attn_check_surveybl3', 'attn_check_surveybl4']
GetTotalScore(df_raw,attnchecks,'AttnCheckTotal')
failedAttnCheck = []
for index,row in df_raw.iterrows(): 
    if row['AttnCheckTotal']<4:
        failedAttnCheck.append(df.loc[index,'record_id'])

### Check who gave different answers for their race/age at start and end of survey

failed_new_qc = []

# Filter out rows where 'race_qc' or 'age_qc' is NaN
df_finished = df[~(df['race_qc'].isna() | df['age_qc'].isna())]

#replace strings
df_finished.loc[df_finished['age_qc'] == '30yrs', 'age_qc'] = 30



#make sure they are both simple integers and so match in type
df_finished.loc[:,'age_qc'] = df_finished['age_qc'].astype(int)
df_finished.loc[:,'age_v2'] = df_finished['age_v2'].astype(int)

# Check if they are giving different answers to race/age at start vs finish:
for index, row in df_finished.iterrows():
    racediff = (abs(row['race_qc'] - row['race_v2']))
    agediff = (abs(row['age_qc'] - row['age_v2']))

    # If both race and age are different, or if one is off by more than 2, add to list
    if ((racediff>0) and (agediff>0)) or ((racediff>1) or (agediff>1)):
        failed_new_qc.append(row['record_id'])


### Check who is giving inconsistent answers about psychedelic use
failed_sp_qc = []
repeat_nomic_q = []
meant_micro = []
failed_micro_qc = []
failed_usetime_qc = []
illogical_year = []
illogical_6month = []
illogical_life = []
wrong_days = []
wrong_recent = []


df_finished = df_raw[df_raw['psycheduse_life_nomic']>0] # Only review for people who have never used a psychedelic. 

# check if there are nan resposnes that likely indicate they just need to redo questionnaires in a survey -- remove them from screening for inconsistencies
nanresponses = df_finished[(df_finished["sp_type_recent"].isna()) | (df_finished["psycheduse_month_nomic"].isna()) | (df_finished["sp_dayslastuse"].isna())]['record_id'].tolist()

nanresponses_screen = df_finished[df_finished["sp_lastuse_days_screen"].isna()]['record_id'].tolist()

df_finished = df_finished[~(df_finished['record_id'].isin(nanresponses+nanresponses_screen))]

absurdity_reasons = {}

#### Check for inconsistencies in psychedelic use answers
for index, row in df_finished.iterrows():
    record = row['record_id']
    ### Check of the've already been asked to verify their answers. -- newrecords already excludes ppl asked to verify who havent finished or havent been scored by max so just check if we need to fail them
    if row['verify_emailed'] == 1:
        # If they have started the verification process, check to see if Max has passed or failed them
        if row["sp_verify_pass"] > 0:
            # If they have been passed, then they are good to go
            continue
        elif row["sp_verify_pass"] < 1:
            # If they have been failed, add to failed_sp_qc
            failed_sp_qc.append(record)
            absurdity_reasons[record] = "Gave inconsistent answers about SP use twice (and/or answers were too different between the two surveys)"
        else:
            # If Max hasn't reviewed then skip them for now
            print("THE CODE IS NOT WORKING AND WE HAVE SOMEONE FOR WHOM sp_verify_pass IS NA -- PLEASE EMAIL MAX TO FIX THIS")
            exit()
            
    ### if they have not yet needed to be verified, then check their answers
    else:
        if (row['psycheduse_yn'] == 1):
            #Check to see if they said they don't use SPs in main survey after saying they did in screening
            if row['psycheduse_life_nomic'] < 1 and row['psychedelicuse_lifetimetot'] < 1:
                failed_sp_qc.append(record)
                absurdity_reasons[record] = f"Has no SP use in main survey despite saying they used SPs in screening"

            ### Find people who give different answers for the most recent SP they used
            if row['sp_type_recent'] != row['sp_type_recent_qc']:
                wrong_recent.append(record)
                absurdity_reasons[record] = f"Has different answers for the most recent SP used in screening vs main survey"
                
            ### Get people who are giving straight ridiculous/fradulent answers about SP use
            # People who report regular non-SP effects from SPs
            for x in range(1, 6):
                if row[f"sp_fraud_aes___{x}"] > 0:
                    failed_sp_qc.append(record)
                    absurdity_reasons[record] = f"Reported non-SP effects from SPs"
                    break

            # People who report a bizarre route of administration 
            if (row['sp_fraud_psi'] < 6) or (row['sp_fraud_lsd'] < 5) or (row['sp_fraud_mesc'] < 6) or ((row['sp_fraud_dmt'] < 6) and (row['sp_fraud_dmt'] > 1)) or (row['sp_fraud_5meo'] < 5):
                failed_sp_qc.append(record)
                absurdity_reasons[record] = f"Reported a bizarre route of administration for SPs"

            #Check if they said they used nonmicrodoses, put 0 non microdoses and 0 microdoses but gave a real answer to total SPs 
            # (indicating they just need to redo the psycheduse_life_nomic Q)
            if row['psycheduse_life_nomic'] < 1 and row['microdose_yn'] < 1 and row['psychedelicuse_lifetimetot'] > 0:
                repeat_nomic_q.append(record)
                absurdity_reasons[record] = f"Has no non-microdose SP use in main survey despite saying they used SPs in screening"

            #Check if they say they seem to be saying theyve only microdosed despite saying nonzero microdoses in screening 
            if row['psycheduse_life_nomic'] < 1 and row['microdose_yn'] > 0 and row['psychedelicuse_lifetimetot'] > 0:
                meant_micro.append(record)
                absurdity_reasons[record] = f"Has no non-microdose SP use in main survey despite saying they used SPs in screening"

            if (row['psycheduse_year_nomic'] < row['psycheduse_6month_nomic']) or (row['psycheduse_year_nomic'] < row['psycheduse_month_nomic']):
                illogical_year.append(record)
                absurdity_reasons[record] = f"Has illogical answers for SP use in the past year"

            if (row['psycheduse_life_nomic'] < row['psycheduse_6month_nomic']) or (row['psycheduse_life_nomic'] < row['psycheduse_month_nomic']) or (row['psycheduse_life_nomic'] < row['psycheduse_year_nomic']):
                illogical_life.append(record)
                absurdity_reasons[record] = f"Has illogical answers for SP use in their lifetime"

            if row['psycheduse_6month_nomic'] < row['psycheduse_month_nomic']:
                illogical_6month.append(record)
                absurdity_reasons[record] = f"Has illogical answers for SP use in the past 6 months"

            # # Check if the days since last use in screening differ from main survey significanctly
            # if pd.notna(row['sp_lastuse_days_screen']):
            #     if abs(row['sp_lastuse_days_screen'] - row['sp_dayslastuse']) > 30:
            #         wrong_days.append(record)
                    # absurdity_reasons[record] = f"Has a large difference in days since last SP use between screening and main survey"

            #Check if the # of uses in a certain period and the days since last use don't add up
            if (row['sp_dayslastuse'] < 30 and row['psycheduse_month_nomic'] < 1) or (row['sp_dayslastuse'] > 30 and row['psycheduse_month_nomic'] > 0):
                failed_usetime_qc.append(record)
                absurdity_reasons[record] = f"Has inconsistent answers for SP use in the past month"
            elif (row['sp_dayslastuse'] < 180 and row['psycheduse_6month_nomic'] < 1) or (row['sp_dayslastuse'] > 185 and row['psycheduse_6month_nomic'] > 0):
                failed_usetime_qc.append(record)
                absurdity_reasons[record] = f"Has inconsistent answers for SP use in the past 6 months"
            elif (row['sp_dayslastuse'] < 365 and row['psycheduse_year_nomic'] < 1) or (row['sp_dayslastuse'] > 365 and row['psycheduse_year_nomic'] > 0):
                failed_usetime_qc.append(record)
                absurdity_reasons[record] = f"Has inconsistent answers for SP use in the past year"
        elif (row['psycheduse_yn'] == 3):
            if row['microdose_yn'] < 1:
                failed_micro_qc.append(record)
                absurdity_reasons[record] = f"Has no microdose SP use in main survey despite saying they used SPs in screening"

        
### Check who did not do the SMS verification
no_sms_verification =df_raw[df_raw['phone_verified'].isna()]['record_id'].tolist()


### Check who did not do the main survey questionnaire
nosurvey1 = df_raw[df_raw['si_2_v2'].isna()]['record_id'].tolist()

########################################################################################################################


#############################
### ACH QC ####
#############################

def validate_retrieved_task_data(json_string, record_id, task_name, fraud_list):
    """
    Validates retrieved task data and handles user interaction for questionable strings.
    
    Args:
        json_string: The string to validate
        record_id: The participant's record ID
        task_name: Name of the task (e.g., "ACH", "VCH", "PRL", "SpaceJunk")
        fraud_list: List to append record_id to if marked as fraud
    
    Returns:
        tuple: (should_process, should_skip)
        - should_process: True if data should be processed normally
        - should_skip: True if record should be skipped entirely
    """
    # Check if it's a proper JSON string (should start with compressed data signature)
    if json_string.startswith("H4sIAAAAA"):
        return True, False
    
    # Check if it's obviously not task data (too short, clearly random text, etc.)
    if len(json_string) < 50 or json_string in ["testing", "test", "hello", "123"]:
        fraud_list.append(record_id)
        print(f"Record {record_id}: Automatically flagged as fraud due to obviously invalid {task_name} data")
        return False, False
    
    # For ambiguous cases, ask the user
    print(f"\nRecord {record_id} has potentially invalid {task_name} retrieved data.")
    print(f"Their input string looks like: '{json_string[:100]}{'...' if len(json_string) > 100 else ''}'")
    print(f"Valid task data should ideally start with 'H4sIAAAAA' (compressed JSON).")
    
    while True:
        user_input = input(f"Does this look like valid task data? (yes/no): ").lower().strip()
        
        if user_input in ['yes', 'y']:
            print(f"PLEASE TELL MAX TO CHECK RECORD {record_id}'s data for {task_name} AND DECIDE HOW TO HANDLE")
            return False, True  # Skip processing but don't mark as fraud
        elif user_input in ['no', 'n']:
            fraud_list.append(record_id)
            print(f"Record {record_id} marked as fraud for invalid {task_name} data")
            return False, False
        else:
            print("Please enter 'yes' or 'no'")



#### Create full trial-by-trial dataset and get useful values added to og dataset
def get_ch_trial_data(dataframe,task_data_taskname,task_data_taskname_alt):
    participant_dfs = []

    for index, row in dataframe.iterrows():
        # Extract the JSON string from the column
        if isinstance(row[task_data_taskname],str):
            compressed_json = row.get(task_data_taskname, "")

            # Check to makse sure the string is not empty
            if compressed_json:
                
                # If the string is "testing", skip this record
                if compressed_json == "testing":
                    continue
                else:


                    try:
                        record = row.get('record_id',"")


                        #decompress and decode the JSON
                        decoded_data = base64.b64decode(compressed_json)
                        decompressed_json = gzip.decompress(decoded_data).decode('utf-8')

                        #Parse JSON data
                        data = json.loads(decompressed_json)

                        #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
                        block_dfs = []
                        blocks = ['component_1','component_2','component_3','component_4']
                        block_num = 1
                        for block in blocks:
                            df1 = pd.DataFrame()
                            df1['response'] = data[block]['response']
                            df1['rt'] = data[block]['responseTime']
                            df1['decibels'] = data[block]['decibels']
                            df1['component'] = block_num
                            df1['record_id']=data['recordId']

                            block_dfs.append(df1)
                            block_num+=1
                        
                        #add all the blocks into one dataframe
                        participant_df = pd.concat(block_dfs,ignore_index=True)

                        #create new column based on decibels for simple % of threshold
                        participant_df['intensity'] = participant_df['decibels'].replace({data['processedData']['intensities']['c25']:25,
                        data['processedData']['intensities']['c50']:50,
                        data['processedData']['intensities']['c75']:75
                        })

                        #add each participant's dataframe into list of dataframes
                        participant_dfs.append(participant_df)

                    except Exception as e:
                        print(f"\nError processing record {record}: {e}\nTheir string:{compressed_json}")
                        continue
        
        ### IF THEY DONT HAVE THE TASK DATA CHECK TO SEE IF THEY USED THE RETRIEVAL METHOD!!!!
        elif isinstance(row[task_data_taskname_alt],str):
            json_string = row.get(task_data_taskname_alt, "")
            if json_string:

                # Validate the retrieved task data
                should_process, should_skip = validate_retrieved_task_data(
                json_string, record, "VCH", fraud_copy_paste_vch
                )
                
                if should_skip:
                    continue
                elif not should_process:
                    continue

                try:
                    #get the record_id
                    record = row.get('record_id',"")

                    #Parse JSON data
                    data = json.loads(json_string)

                    #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
                    block_dfs = []
                    blocks = ['component_1','component_2','component_3','component_4']
                    block_num = 1
                    for block in blocks:
                        df1 = pd.DataFrame()
                        df1['response'] = data[block]['response']
                        df1['rt'] = data[block]['responseTime']
                        df1['decibels'] = data[block]['decibels']
                        df1['component'] = block_num
                        df1['record_id']=record

                        block_dfs.append(df1)
                        block_num+=1
                    
                    #add all the blocks into one dataframe
                    participant_df = pd.concat(block_dfs,ignore_index=True)

                    #create new column based on decibels for simple % of threshold
                    participant_df['intensity'] = participant_df['decibels'].replace({data['processedData']['intensities']['c25']:25,
                    data['processedData']['intensities']['c50']:50,
                    data['processedData']['intensities']['c75']:75
                    })

                    #add each participant's dataframe into list of dataframes
                    participant_dfs.append(participant_df)
                except (json.JSONDecodeError, KeyError) as e:
                    print(f"\nError processing record {record}: {e}\nreplay_email:{df.loc[df['record_id']==record,'replay_email'].values[0]}\nThey pasted this string:{json_string}")
                    fraud_copy_paste_ach.append(record)
                    continue
                    

    # concatenate each participant's full CH data into one ENORMOUS dataframe
    ch_data_all = pd.concat(participant_dfs,ignore_index=True)

    return ch_data_all



### Get trial-by-trial dataframe for the entire sample
ach_master = get_ch_trial_data(df_nocopypaste,'task_data_ach_task_short_baseline','task_data_ach_bl_retrieved')
ach_master['record_id']= ach_master['record_id'].astype('int64')
ach_master['trial'] = ach_master.groupby('record_id').cumcount() + 1

### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
ach_master_pivoted = ach_master.pivot(index='record_id', columns='trial', values='rt')#record_id becomes the index
# Find duplicated rows
duplicated_rows = ach_master_pivoted[ach_master_pivoted.duplicated(keep=False)]
# Extract the record_ids of duplicated rows 
fraud_copy_paste_ach = duplicated_rows.index.tolist() 

### First, make sure detection slope is positive and non-zero
def test_detection_probability(data,intensity_var):
    results = []
    for record_id, group in data.groupby('record_id'):
        model = smf.logit(formula=f'response ~ {intensity_var}', data=group).fit(disp=0)
        p_value = model.pvalues[f'{intensity_var}']
        beta_coefficient = model.params[f'{intensity_var}']
        results.append({'record_id': record_id, 'p_value': p_value, 'beta_coefficient': beta_coefficient})
    
    results_df = pd.DataFrame(results)
    return results_df

detections = ach_master[['record_id','decibels','response']]
detections = detections.dropna()
detection_slopes = test_detection_probability(detections,'decibels')

### Second, test whether they have <50% detection of first 15 trials
df_15 = ach_master[ach_master['trial']<16]
detection_15 = df_15.groupby('record_id').mean().reset_index()


#Get those who pass and fail qc
non_negative = detection_slopes[detection_slopes['beta_coefficient']>0]['record_id'].tolist()
non_zero = detection_slopes[detection_slopes['p_value']<0.05]['record_id'].tolist()

negative = detection_slopes[detection_slopes['beta_coefficient']<0]['record_id'].tolist()
zero = detection_slopes[detection_slopes['p_value']>0.05]['record_id'].tolist()

fail_first_fifteen = detection_15[detection_15['response']<0.5]['record_id'].tolist()
good_15 = detection_15[detection_15['response']>0.5]['record_id'].tolist()




#############################
### VCH QC ####
#############################
#### Create full trial-by-trial dataset and get useful values added to og dataset
def get_vch_trial_data(dataframe,task_data_taskname,task_data_taskname_alt):
    participant_dfs = []

    for index, row in dataframe.iterrows():
        # Extract the JSON string from the column
        if isinstance(row[task_data_taskname],str):
            compressed_json = row.get(task_data_taskname, "")

        # If the string is "testing", skip this record
            if compressed_json == "testing":
                continue
            else:

                #decompress and decode the JSON
                decoded_data = base64.b64decode(compressed_json)
                decompressed_json = gzip.decompress(decoded_data).decode('utf-8')

                #Parse JSON data
                data = json.loads(decompressed_json)

                #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
                block_dfs = []
                blocks = ['component_1','component_2','component_3','component_4']
                block_num = 1
                for block in blocks:
                    df1 = pd.DataFrame()
                    df1['response'] = data[block]['response']
                    df1['rt'] = data[block]['responseTime']
                    df1['contrasts'] = data[block]['contrasts']
                    df1['component'] = block_num
                    df1['record_id']=data['recordId']

                    block_dfs.append(df1)
                    block_num+=1
                
                #add all the blocks into one dataframe
                participant_df = pd.concat(block_dfs,ignore_index=True)

                ## create new column based on decibels for simple % of threshold

                #Get the contrasts, round them to 5 digits (so replace doesn't freak out), get the contrasts sorted from smallest to largest, and then assign them as just 0,25,50,75
                participant_df['contrasts'] = participant_df['contrasts'].round(5)
                unique_values = participant_df['contrasts'].unique()
                unique_values_list = unique_values.tolist()
                unique_values_list.sort()

                participant_df['intensity'] = participant_df['contrasts'].replace({unique_values_list[1]:25,
                unique_values_list[2]:50,
                unique_values_list[3]:75
                })

                #add each participant's dataframe into list of dataframes
                participant_dfs.append(participant_df)

        ### IF THEY DONT HAVE THE TASK DATA CHECK TO SEE IF THEY USED THE RETRIEVAL METHOD!!!!
        elif isinstance(row[task_data_taskname_alt],str):
            json_string = row.get(task_data_taskname_alt, "")

            record = row.get('record_id',"")

            try:

                #Parse JSON data
                data = json.loads(json_string)

                #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
                block_dfs = []
                blocks = ['component_1','component_2','component_3','component_4']
                block_num = 1
                for block in blocks:
                    df1 = pd.DataFrame()
                    df1['response'] = data[block]['response']
                    df1['rt'] = data[block]['responseTime']
                    df1['contrasts'] = data[block]['contrasts']
                    df1['component'] = block_num
                    df1['record_id']= record

                    block_dfs.append(df1)
                    block_num+=1
                
                #add all the blocks into one dataframe
                participant_df = pd.concat(block_dfs,ignore_index=True)

                ## create new column based on decibels for simple % of threshold

                #Get the contrasts, round them to 5 digits (so replace doesn't freak out), get the contrasts sorted from smallest to largest, and then assign them as just 0,25,50,75
                participant_df['contrasts'] = participant_df['contrasts'].round(5)
                unique_values = participant_df['contrasts'].unique()
                unique_values_list = unique_values.tolist()
                unique_values_list.sort()

                participant_df['intensity'] = participant_df['contrasts'].replace({unique_values_list[1]:25,
                unique_values_list[2]:50,
                unique_values_list[3]:75
                })

                #add each participant's dataframe into list of dataframes
                participant_dfs.append(participant_df)
            except (json.JSONDecodeError, KeyError) as e:
                print(f"\nError processing record {record}: {e}\nreplay_email:{df.loc[df['record_id']==record,'replay_email'].values[0]}\nThey pasted this string:{json_string}")
                fraud_copy_paste_vch.append(record)
                continue

    # concatenate each participant's full CH data into one ENORMOUS dataframe
    ch_data_all = pd.concat(participant_dfs,ignore_index=True)

    return ch_data_all

vch_master = get_vch_trial_data(df_nocopypaste,'task_data_vch_short_psychedelic_bl','task_data_vch_bl_retrieved')
vch_master['record_id']= vch_master['record_id'].astype('int64')
vch_master['trial'] = vch_master.groupby('record_id').cumcount() + 1


### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
vch_master_pivoted = vch_master.pivot(index='record_id', columns='trial', values='rt')#record_id becomes the index
# Find duplicated rows
duplicated_rows_vch = vch_master_pivoted[vch_master_pivoted.duplicated(keep=False)]
# Extract the record_ids of duplicated rows 
fraud_copy_paste_vch = duplicated_rows_vch.index.tolist() 

### Test for nonzero positive slope
detections = vch_master[['record_id','contrasts','response']]
detections = detections.dropna()
detection_slopes = test_detection_probability(detections,'contrasts')

### Second, test whether they have <50% detection of first 15 trials
df_15 = vch_master[vch_master['trial']<16]
detection_15 = df_15.groupby('record_id').mean().reset_index()


#Get those who pass and fail qc
non_negative_vch = detection_slopes[detection_slopes['beta_coefficient']>0]['record_id'].tolist()
non_zero_vch = detection_slopes[detection_slopes['p_value']<0.05]['record_id'].tolist()

negative_vch = detection_slopes[detection_slopes['beta_coefficient']<0]['record_id'].tolist()
zero_vch = detection_slopes[detection_slopes['p_value']>0.05]['record_id'].tolist()

fail_first_fifteen_vch = detection_15[detection_15['response']<0.5]['record_id'].tolist()
good_15_vch = detection_15[detection_15['response']>0.5]['record_id'].tolist()

#############################
### PRL QC ####
#############################
# Function that converts PRL JSON string into dataframe
def process_json_string(compressed_str):
    decoded_bytes = base64.b64decode(compressed_str)
    decompressed_str = gzip.decompress(decoded_bytes).decode('utf-8')

    data_dict = json.loads(decompressed_str)
    
    # Extract 'data' key, record id, and project id
    data_array = data_dict.get("data", [])
    record_id = data_dict.get('recordId', 'N/A')
    project_id = data_dict.get('projectId', 'N/A')
    
    # Add 'record_id' and 'projectId' to each row in data_array
    for row in data_array:
        row['record_id'] = record_id
        row['projectId'] = project_id
    
    return data_array

### Convert all of the JSON strings into dataframes and then concatenate together. 
# List to hold DataFrames
dataframes_prl_list = []
# For each subject, decompress the JSON string, convert to dataframe, add to list of dataframes
for row in range(len(df_nocopypaste)):
    if isinstance(df_nocopypaste.loc[row,'task_data_prltask'],str):
        compressed_string_prl = df_nocopypaste.loc[row,'task_data_prltask']
        
        # If the string is "testing", skip this record
        if compressed_string_prl == "testing":
            continue
        else:
            decompressed_prl_data_array = process_json_string(compressed_string_prl)
            #convert to PANDAS dataframe
            decompressed_prl_df = pd.DataFrame(decompressed_prl_data_array)
            #append dataframe to list of dataframes
            dataframes_prl_list.append(decompressed_prl_df)

    #If they don't have the prl task data check to see if they used the backup option:
    elif isinstance(df_nocopypaste.loc[row,'task_data_prl_bl_retrieved'],str):
        record = df_nocopypaste.loc[row,'record_id']
        json_string = df_nocopypaste.loc[row,'task_data_prl_bl_retrieved']
        
        # Validate the retrieved data
        should_process, should_skip = validate_retrieved_task_data(
            json_string, record, "PRL", fraud_copy_paste_prl
        )
        
        if should_skip:
            continue
        elif not should_process:
            continue
            
        try:
            data_array = json.loads(json_string)
            if isinstance(data_array,dict):
                data_array['record_id'] = record
                dataframes_prl_list.append(data_array)
            else:
                print(f"WARNING! PRL data for {record} is just {data_array} -- marking them as a scammer for now.")
                fraud_copy_paste_prl.append(record)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"\nError processing record {record}: {e}")
            fraud_copy_paste_prl.append(record)
            continue

#Concatenate dataframes
df_prl = pd.concat(dataframes_prl_list, ignore_index=True)

#replace fractal1 etc. with just 1,2, and 3
df_prl = df_prl.replace('fractal1', 1)
df_prl = df_prl.replace('fractal2', 2)
df_prl = df_prl.replace('fractal3', 3)

#convert the fractal numbers to integers
df_prl['choice'] = df_prl['choice'].astype(int)
# at some point may want to replace -999 with NaN df_prl = df_prl.replace(-999,np.nan)

#convert record_id to integer to match df
df_prl['record_id'] = df_prl['record_id'].astype(int)

df_prl.reset_index(drop=True,inplace=True)

# #Replace -999 with NaNs
# df_prl = df_prl.replace(-999,np.nan)

### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
try:
    prl_master_pivoted = df_prl.pivot(index='record_id', columns='trial', values='decisionTime')#record_id becomes the index
    # Find duplicated rows
    duplicated_rows_prl = prl_master_pivoted[prl_master_pivoted.duplicated(keep=False)]
    # Extract the record_ids of duplicated rows 
    fraud_copy_paste_prl.extend(duplicated_rows_prl.index.tolist())
except Exception as e:
    print(f"Error in pivoting PRL data: {e}")
    # fraud_copy_paste_prl = ["nothing"]

#####This function takes the number of trials that meet a certain criteria and then appends a column to the dataframe w/%of total trials meeting those criteria
def get_percent_trials_per_participant(dataframe,colname,colvaluetocount,newcolname):
    number_trials = dataframe[dataframe[colname]==colvaluetocount].groupby('record_id').size()
    total_trials = dataframe.groupby('record_id').size()
    percent_trials = (number_trials/total_trials) * 100
    percent_trials_df = percent_trials.reset_index(name=newcolname)
    newdataframe = dataframe.merge(percent_trials_df,on='record_id')
    return newdataframe
#you need to set it equal to do "dataframe = get_percent_trials_per_participant" to actually modify the dataframe


#Calculate the % of trials that each participant is picking the right answer and appends as column:
df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='rewardProbChoice',colvaluetocount=0.85,newcolname='%Correct')

##Which participants are performing worse than chance? 
grouped_record_id = df_prl.groupby('record_id')['%Correct'].mean().reset_index()
worse_than_chance = []
for row in range(len(grouped_record_id)):
    if grouped_record_id.loc[row,'%Correct']<34:
        worse_than_chance.append(grouped_record_id.loc[row,'record_id'])

##Which participants are not responding 10% of the time? 
#Calculate the % of trials that each participant is not responding
df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='keyChoice',colvaluetocount=-999,newcolname='%NoResponse')
grouped_record_id = df_prl.groupby('record_id')['%NoResponse'].mean().reset_index()
#now find who doesn't respond >10% of them time
non_responders = []
for row in range(len(grouped_record_id)):
    if grouped_record_id.iloc[row]['%NoResponse']>10:
        non_responders.append(grouped_record_id.loc[row,'record_id'])

##Determine whether each trial represents a win-stay or a lose shift
win_stay_lose_stay = []
for row in range(len(df_prl)):
    if (row-1) <0 or df_prl.loc[row,'record_id']!=df_prl.loc[row-1,'record_id'] or np.isnan(df_prl.loc[row,'outcome']) or df_prl.loc[(row-1),'outcome']==-999:
        win_stay_lose_stay.append(np.nan)    
    else:
        if df_prl.loc[(row-1),'outcome']==1 and df_prl.loc[row,'rewardProbChoice']==df_prl.loc[(row-1),'rewardProbChoice']:
            win_stay_lose_stay.append('win-stay')
        elif df_prl.loc[(row-1),'outcome']==1 and df_prl.loc[row,'rewardProbChoice']!=df_prl.loc[(row-1),'rewardProbChoice']:
            win_stay_lose_stay.append('win-switch')
        elif df_prl.loc[(row-1),'outcome']==0 and df_prl.loc[row,'rewardProbChoice']!=df_prl.loc[(row-1),'rewardProbChoice']:
            win_stay_lose_stay.append('lose-switch')
        elif df_prl.loc[(row-1),'outcome']==0 and df_prl.loc[row,'rewardProbChoice']==df_prl.loc[(row-1),'rewardProbChoice']:
            win_stay_lose_stay.append('lose-stay')
        else:
            print(df_prl.loc[row,'trial'])


#Add this as a column            
df_prl['win_stay_lose_switch'] = win_stay_lose_stay
## Get the percent win- and lose-stays
df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='win_stay_lose_switch',colvaluetocount='win-stay',newcolname='win_stay_percent')
df_prl = get_percent_trials_per_participant(df_prl,'win_stay_lose_switch','lose-stay','lose_stay_percent')

######## DISCUSS WITH AL AND ALEXANDRIA; DOES THIS MAKE SENSE TO KEEP? 
##Which participants do not exhibit ANY (less than 1% or <3 trials) lose-stay behavior, suggesting they did not understand the instructions?
no_lose_stay = []
grouped_record_id = df_prl.groupby('record_id')['lose_stay_percent'].mean().reset_index()
for row in range(len(grouped_record_id)):
    if grouped_record_id.loc[row,'lose_stay_percent']<1:
        no_lose_stay.append(grouped_record_id.loc[row,'record_id'])



#############################
### SpaceJunk QC ####
#############################

### Convert all of the JSON strings into dataframes and then concatenate together. 
# List to hold DataFrames
dataframes_spacejunk_list = []
# For each subject, decompress the JSON string, convert to dataframe, add to list of dataframes
for row in range(len(df_nocopypaste)):
    if isinstance(df_nocopypaste.loc[row,'task_data_spacejunk_bl'],str):
        compressed_string = df_nocopypaste.loc[row,'task_data_spacejunk_bl']

        # If the string is "testing", skip this record
        if compressed_string == "testing":
            continue
        else:
            decompressed_data_array = process_json_string(compressed_string)
            #convert to PANDAS dataframe
            decompressed_spacejunk_df = pd.DataFrame(decompressed_data_array)
            #append dataframe to list of dataframes
            dataframes_spacejunk_list.append(decompressed_spacejunk_df)
    
    #If they don't have the main spacejunk task saved, check to see if they have the backup saved:
    elif isinstance(df_nocopypaste.loc[row,'task_data_spacejunk_bl_retrieved'],str):
        record = df_nocopypaste.loc[row,'record_id']
        json_string = df_nocopypaste.loc[row,'task_data_spacejunk_bl_retrieved']
        
        # Validate the retrieved data
        should_process, should_skip = validate_retrieved_task_data(
            json_string, record, "SpaceJunk", fraud_copy_paste_spacejunk
        )
        
        if should_skip:
            continue
        elif not should_process:
            continue
            
        try:
            data_array = json.loads(json_string)
            data_array['record_id'] = record
            dataframes_spacejunk_list.append(data_array)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"\nError processing record {record}: {e}")
            fraud_copy_paste_spacejunk.append(record)
            continue

#Concatenate dataframes
df_spacejunk = pd.concat(dataframes_spacejunk_list, ignore_index=True)

#convert record_id to integer to match df
df_spacejunk['record_id'] = df_spacejunk['record_id'].astype(int)

#Add trials of game
df_spacejunk['trial'] = df_spacejunk.groupby('record_id').cumcount() + 1

#Add trials of each level
df_spacejunk['trial_thislevel'] = df_spacejunk.groupby(['record_id','level']).cumcount() + 1


### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
spacejunk_master_pivoted = df_spacejunk.pivot(index='record_id', columns='trial', values='timeToComplete')#record_id becomes the index
# Find duplicated rows
duplicated_rows_spacejunk = spacejunk_master_pivoted[spacejunk_master_pivoted.duplicated(keep=False)]
# Extract the record_ids of duplicated rows 
fraud_copy_paste_spacejunk = duplicated_rows_spacejunk.index.tolist() 

#### Find out which yale students screen positive for mental health issue and require an email
al_yale_student_mentalhealth_psychosis = df_raw_yale[df_raw_yale["chat_distress"]>0]['record_id'].tolist()
al_yale_student_mentalhealth_substances = df_raw_yale[df_raw_yale["problematicuse"]>0]['record_id'].tolist()
al_yale_student_mentalhealth_mdd_si = df_raw_yale[(df_raw_yale["phq9_tot"]>19) | (df_raw_yale["phq9"]>0)]['record_id'].tolist()

#### One last thing! For the Yale records -- since we let everyone through now, go check whether they are even eligibile
ineligibile_yalies = []

if len(rec_to_check_yale)>0:
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['age_v2']>65,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['age_v2']<18,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['cognition_screener_v2']>0,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['seizure_hx_v2']>0,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['intox_screen_v2']>0,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['raven_total_score_v2']<1,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<28 & (df_raw_yale['cannabis_frequency']>9),'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<14 & (df_raw_yale['cannabis_frequency'].isin([8,9])),'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<7 & (df_raw_yale['cannabis_frequency']==7),'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<3,'record_id'].tolist())
    ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['no_computer']>0,'record_id'].tolist())




###############################
##############################
#### Create Master QC lists
passed_qc = []

master_qc_fail = (
    zero +
    failedAttnCheck +
    negative +
    fail_first_fifteen +
    zero_vch +
    negative_vch +
    fail_first_fifteen_vch +
    failed_sp_qc +
    failed_micro_qc+
    repeat_nomic_q +
    meant_micro +
    no_lose_stay +
    failed_usetime_qc +
    non_responders +
    worse_than_chance +
    nosurvey1 +
    no_sms_verification +
    illogical_year +
    illogical_life +
    illogical_6month +
    no_prl +
    no_ach +
    no_vch +
    failed_new_qc +
    wrong_recent
)

yale_qc_fail = (
    failedAttnCheck +
    failed_sp_qc +
    failed_micro_qc+
    repeat_nomic_q +
    meant_micro +
    nosurvey1 +
    illogical_year +
    illogical_life +
    illogical_6month +
    fraud_copy_paste_prl +
    non_responders +
    worse_than_chance +
    no_lose_stay + 
    # fraud_copy_paste_ach +
    # no_spacejunk +
    # no_ach +
    no_vch +
    no_prl +
    failed_new_qc +
    ineligibile_yalies
)

yale_qc_fail = [x for x in yale_qc_fail if x not in rec_to_check_reg]
        

######################################################################################################
################# QC LOOP!!!! ########################################################################
######################################################################################################

expensesheets = []

# Create a dataframe to modify in order to send updates to redcap -- and convert replays to int objects 
update_df = df_og[df_og['record_id'].isin(records_to_check)].copy()

# Get a dataframe for the json strings of failed tasks
jsons_df = pd.read_csv(json_path)
# Read in digitalcard_template for folks who need an extra sheet for digital gift cards
digitalcard_template_path = "/Volumes/psychedelics/digitalcard_template.csv"
digitalcard_template = pd.read_csv(digitalcard_template_path)

# Limit to the columns we modify 
updatecols = [
    # Critical failures
    'qc_passed','qc_notes',
    
    # Task data columns
    'task_data_ach_task_short_baseline','task_data_vch_short_psychedelic_bl', 'task_data_prltask',

    # _complete columns
    'task_data_ach_task_short_baseline_complete','task_data_vch_short_psychedelic_bl_complete', 'task_data_prltask_complete',

    
    # Qualifying columns
    'task_data_auditory_qualifying_task_2','task_data_visual_qualifying_task_2','task_data_auditory_qualifying','task_data_visual_qualifying',
    
    # Replay counters
    'ach_replay', 'ach_replay_4', 'ach_replay_2', 'ach_replay_3',
    'vch_replay', 'vch_replay_4', 'vch_replay_2', 'vch_replay_3',
    'prl_replay', 'prl_replay_4', 'prl_replay_2', 'prl_replay_3',
    
    # Verification
    'verify_emailed',

    # dumb payment info
    "send_pay_confirm","pay_day_ofweek","employee_name",

    # Addon columns
    "addons_completed", "addons_payment",

    # Yale mental health emails
    "al_yale_student_mentalhealth_mdd_si",
    "al_yale_student_mentalhealth_psychosis",
    "al_yale_student_mentalhealth_substances",
    
    # Email template fields
    'replay_links_ach',
    'replay_links_ach_2',
    'replay_links_ach_3',
    'replay_links_ach_4',
    'replay_links_vch',
    'replay_links_vch_2',
    'replay_links_vch_3',
    'replay_links_vch_4',
    'replay_links_prl',
    'replay_links_prl_2',
    'replay_links_prl_3',
    'replay_links_prl_4',
    'fraudulent_email_inconsistentanswers',
    'fraudulent_email',
    'inconsistent_sp_answers',
    'ip_zoom_invite',
    'longitudinal_waiting_confirmation_better',
    'longitudinal_continue_link',
    'eligible_notify',
    'ineligibile_fraud',
    'zoom_eligibility',
    'fourth_fail'
] + task_instruments['auditory_ch_task'] + task_instruments['visual_ch_task'] + task_instruments['prl_task']

# Remove the fields that are floats and we don't want to modify
updatecols = [col for col in updatecols if "exceed" not in col]
updatecols = [col for col in updatecols if "exceed" not in col]

##############################################################################################################################################################

# =====================
# Configuration Mappings
# =====================
BASE_PATH = sharedrive_network_path
QC_TODO_PATH = f"{BASE_PATH}/qc_to_dos"
os.makedirs(QC_TODO_PATH, exist_ok=True)
expensesheet_path = f"{BASE_PATH}/expense_sheets"
# Path that we will save everthing to:
date_directory = f"{QC_TODO_PATH}/{datetime.now().strftime('%Y-%m-%d')}/"
os.makedirs(date_directory, exist_ok=True)

TASK_CONFIG = {
    'auditory_ch_task': {
        'failure_conditions': ['zero', 'negative'],
        'task_col': 'task_data_ach_task_short_baseline',
        'replay_fields': ['ach_replay', 'ach_replay_2', 'ach_replay_3'],
        'qualifying_cols': ["task_data_ach_task_short_baseline_complete","task_data_auditory_qualifying_task_2", "task_data_visual_qualifying_task_2"],
        'url_col': 'ach_url',
        'instruments': task_instruments['auditory_ch_task'],
        'templates': {
            1: 'replay_links_ach.emltpl',
            2: 'replay_links_ach_2.emltpl',
            3: 'replay_links_ach_3.emltpl',
            4: 'replay_links_ach_4.emltpl'
        },
        'reset_cols': [
            'headphones_recheck_ah', 'headphones_type_ah', 'monitor_recheck_ah', 'monitor_type_ah',
            'browser_ach', 'ach_vol_adj_yn', 'ach_vol_adj_amnt',
            'task_nosave_yn_ach_bl', 'task_nosave_opt_ach_bl', 'task_nosave_ach_bl',
            'task_data_ach_bl_retrieved', 'confirm_ach_tasks'
        ]
    },
    'visual_ch_task': {
        'failure_conditions': ['zero_vch', 'negative_vch'],
        'task_col': 'task_data_vch_short_psychedelic_bl',
        'replay_fields': ['vch_replay', 'vch_replay_2', 'vch_replay_3'],
        'qualifying_cols': ["task_data_auditory_qualifying", "task_data_visual_qualifying","task_data_vch_short_psychedelic_bl_complete"],
        'url_col': 'vch_url',
        'instruments': task_instruments['visual_ch_task'],
        'templates': {
            1: 'replay_links_vch.emltpl',
            2: 'replay_links_vch_2.emltpl',
            3: 'replay_links_vch_3.emltpl',
            4: 'replay_links_vch_4.emltpl'
        },
        'reset_cols': [
            'headphones_check', 'monitor_check', 'browser_vch', 'private_place_vch',
            'task_nosave_yn_vch_bl', 'task_nosave_opt_vch_bl', 'task_nosave_vch_bl',
            'task_data_vch_bl_retrieved', 'confirm_vch_tasks'
        ]
    },
    'prl_task': {
        'failure_conditions': ['no_lose_stay', 'non_responders', 'worse_than_chance'],
        'task_col': 'task_data_prltask',
        'replay_fields': ['prl_replay', 'prl_replay_2', 'prl_replay_3'],
        'qualifying_cols': ["task_data_prltask_complete"],
        'url_col': 'prl_url',
        'instruments': task_instruments['prl_task'],
        'templates': {
            1: 'replay_links_prl.emltpl',
            2: 'replay_links_prl_2.emltpl',
            3: 'replay_links_prl_3.emltpl',
            4: 'replay_links_prl_4.emltpl'
        },
        'reset_cols': [
            'browser_prl', 'full_screen', 'task_nosave_yn_prlbl', 'task_nosave_opt_prlbl',
            'task_nosave_prlbl', 'task_data_prl_bl_retrieved', 'confirm_prl_complete'
        ]
    },
    'spacejunk_game': {
        'failure_conditions': [],
        'task_col': 'task_data_spacejunk_bl',
        'replay_fields': [],
        'qualifying_cols': ["task_data_spacejunk_bl_complete"],
        'url_col': 'spacejunk_url',
        'instruments': task_instruments['spacejunk_game'],
        'templates': {},
        'reset_cols': [
            'monitor_recheck_spacejunk', 'monitor_check_spacejunk', 'browser_spacejunk',
            'full_screen_spacejunk', 'task_nosave_yn_spacejunktaskbl',
            'task_data_spacejunk_bl_retrieved', 'confirm_spacejunktask_complete'
        ]
    }
}

CRITICAL_FAILURES = {
    'failedAttnCheck': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email_inconsistentanswers.emltpl',
        'notes': "failed attention check"
    },
    'failed_new_qc': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email_inconsistentanswers.emltpl',
        'notes': "failed race/age qc"
    },
    'fraud_copy_paste_ach': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email.emltpl',
        'notes': "copy pasted ach"
    },
        'fraud_copy_paste_prl': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email.emltpl',
        'notes': "copy pasted prl"
    },
    'fraud_copy_paste_vch': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email.emltpl',
        'notes': "copy pasted vch"
    },
    'skippedscreen': {
        'special': lambda r: (f"Spydialer check needed for {df.loc[df['record_id']==r, 'phone_number'].values[0]}",
                             f"https://www.spydialer.com")
    },
    'failed_sp_qc': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email_inconsistentanswers.emltpl',
        'notes': "Absurd psychedelics responses"
    },
    'failed_control_qc': {
        'qc_field': 'qc_passed',
        'action': {'qc_passed': 0},
        'template': 'fraudulent_email_inconsistentanswers.emltpl',
        'notes': "Endorsed psychedelic use between screening(where they said they are a control w/o sp hx) and the main questionnaire"
    },
}

# We'll now accumulate strings for each condition instead of DataFrame rows.
skippedscreen_strings = []
critical_failures_strings = []
inconsistent_sp_answers_strings = []
# Set up separate lists for each task failure using a dictionary keyed by "{task}_fail"
task_fail_strings = {f"{task}_fail": [] for task in TASK_CONFIG.keys()}
longitudinalinvites_string = ""

# ====================
# Helper Functions
# ====================

def get_replay_status(df, record, replay_fields):
    record_df = df.loc[df['record_id'] == record]
    if record_df.empty:
        print(f"Warning: record {record} not found for replay status!")
        return np.nan
    record_data = record_df.reset_index(drop=True).iloc[0]
    # Check highest replay first!
    if len(replay_fields) > 2 and pd.notna(record_data[replay_fields[2]]):
        return 3
    elif len(replay_fields) > 1 and pd.notna(record_data[replay_fields[1]]):
        return 2
    elif pd.notna(record_data[replay_fields[0]]):
        return 1
    return np.nan

def update_replay_system(update_df, record, replay_fields, current_status):
    """Updates replay fields with proper sequencing"""
    if pd.isna(current_status):  # First attempt
        update_df.loc[update_df['record_id'] == record, replay_fields[0]] = 1
    elif current_status == 1 and len(replay_fields) > 1:  # Second attempt
        update_df.loc[update_df['record_id'] == record, replay_fields[1]] = 1
    elif current_status == 2 and len(replay_fields) > 2:  # Third attempt
        update_df.loc[update_df['record_id'] == record, replay_fields[2]] = 1
    else:
        print(f"Max replays reached for {record}")

# ====================
# Main Processing Loop
# ====================

# If we have new study completions, then run the loop!
if (len(rec_to_check_reg) > 0) or (len(addon_finishes) > 0) or (len(rec_to_check_yale) > 0):
    update_df = df[df['record_id'].isin(rec_to_check_reg + addon_finishes + rec_to_check_yale)].copy()
    update_df.set_index('record_id', inplace=True)
    update_df = update_df.copy()
    update_df['record_id'] = update_df.index
    json_df = pd.DataFrame()

    # For each record in rec_to_check_reg...
    for record in rec_to_check_reg:
        update_df = update_df.copy()
        record_df = df.loc[df['record_id'] == record].reset_index(drop=True)
        if record_df.empty:
            print(f"Warning: record {record} not found in df; skipping.")
            continue
        current_data = record_df.iloc[0]
        record_updates = {}
        task_processed = False  # Will be set to True if ANY task failure is found

        # ----- Handle Critical Failures first -----
        critical_handled = False
        for failure_type in CRITICAL_FAILURES:
            if record in globals().get(failure_type, []):
                # For non-skippedscreen failures:
                if failure_type != 'skippedscreen':
                    record_updates['qc_passed'] = 0
                    if record in failed_sp_qc:
                        record_updates['qc_notes'] = f"{CRITICAL_FAILURES[failure_type]['notes']} {absurdity_reasons[record]}"
                    else:
                        record_updates['qc_notes'] = CRITICAL_FAILURES[failure_type]['notes']
                    emailname = CRITICAL_FAILURES[failure_type]['template']
                    updatecol = emailname.replace('.emltpl', '')
                    record_updates[f"{updatecol}"]=1
                    
                    # ### Add manual instructions to .txt file
                    # rec_str = (
                    # f"{record}\n" \
                    # f"{current_data.get('replay_email', '')}\n" \
                    # f"{current_data.get('email_addtl_contact', '')}\n" \
                    # # f"{current_data.get('payment_url', '')}\n" \ # We do not need to include the payment url
                    # f"{CRITICAL_FAILURES[failure_type]['template']}\n\n"
                    # )

                    # critical_failures_strings.append(rec_str)
                else:  # For skippedscreen
                    spydialer_check.append(record)

                    # ### Add manual instructions to .txt file
                    # phone = current_data.get('phone_number', '')
                    # rec_str = f"{record}\n" \
                    #         f"{current_data.get('replay_email', '')}\n" \
                    #         f"{current_data.get('email_addtl_contact', '')}\n" \
                    #         f"https://www.spydialer.com\n" \
                    #         f"VOIP check needed for {phone}\n\n"
                    # skippedscreen_strings.append(rec_str)
                critical_handled = True
                # Do not break here; if a record meets multiple critical conditions, add once.
                break  

        if critical_handled:
            # Apply critical failure updates and skip further processing for this record.
            for col, val in record_updates.items():
                update_df.loc[update_df['record_id'] == record, col] = val
            continue

        # ----- Process Verification Flags (Inconsistent SP Answers) -----
        verif_handled = False
        for cond in ['repeat_nomic_q', 'meant_micro', 'failed_usetime_qc', 'wrong_days',
                     'illogical_life', 'illogical_year', 'illogical_6month',"nanresponses","nanresponses_screen","wrong_recent"]:
            if record in globals().get(cond, []):
                # ### Add manual instructions to .txt file
                # if current_data.get('verify_emailed', 0) == 1:
                #     if pd.notna(current_data.get('sp_tot_verify', np.nan)):
                #         rec_str = f"{record}\nClarified answers but still giving illogical answers.\n\n"
                #     else:
                #         rec_str = f"{record}\nAlready emailed but has not fixed their answers.\n\n"
                # else:
                    # record_updates['verify_emailed'] = 1
                    # record_updates['inconsistent_sp_answers'] = 1
                #     rec_str = f"{record}\n" \
                #               f"{current_data.get('replay_email', '')}\n" \
                #               f"{current_data.get('email_addtl_contact', '')}\n" \
                #               f"{current_data.get('clarification_url', '')}\n" \
                #               f"inconsistent_sp_answers.emltpl\n\n"
                # inconsistent_sp_answers_strings.append(rec_str)
                if cond in ["nanresponses"]:
                    record_updates["nan_values_fix"] = 1
                if cond in ["nanresponses_screen"]:
                    record_updates["nan_values_fix_screen"] = 1
                else:
                    record_updates['inconsistent_sp_answers'] = 1
                    record_updates['verify_emailed'] = 1

                verif_handled = True
                break

        # ----- Process Task Failures -----
        # Get all tasks they may have failed if they dont meet above conditions
        for task, config in TASK_CONFIG.items():
            task_failed = False
            for cond in config['failure_conditions']:
                if record in globals().get(cond, []):
                    task_failed = True
                    break
            if task_failed:
                # Update replay counter and task data for this task
                current_status = get_replay_status(df, record, config['replay_fields'])
                update_replay_system(update_df, record, config['replay_fields'], current_status)
                
                record_updates[config['task_col']] = np.nan  # Update task column
                for col in config['qualifying_cols']:
                    record_updates[col] = '<font color="red">Incomplete</font>'
                
                # Set all reset_cols to np.nan for this record
                for col in config.get('reset_cols', []):
                    if col in update_df.columns:
                        update_df.loc[update_df['record_id'] == record, col] = np.nan
                
                next_attempt = current_status + 1 if not pd.isna(current_status) else 1
                
                # Append JSON info (do this for each task failure)
                json_df = pd.concat([json_df, pd.DataFrame([{
                    "record_id": record,
                    "task": task,
                    "fail_reason": next((cond for cond in config['failure_conditions'] 
                                        if record in globals().get(cond, [])), 'unknown').replace('_', ' ').title(),
                    "fail_attempt": next_attempt,
                    "json_string": current_data[config['task_col']]
                }])], ignore_index=True)
                
                # Get the appropriate email template and mark to send in update_df (if this is the fourth fail, then we always do same email)
                template = config['templates'].get(next_attempt, 'fourth_fail.emltpl')
                template_col = template.replace('.emltpl', '')
                update_df.loc[update_df['record_id'] == record, template_col] = 1

                # Also set the _complete task field to incomplete in red
                complete_col = f"{config['task_col']}._complete"
                update_df.loc[update_df['record_id'] == record, complete_col] = '<font color="red">Incomplete</font>'
                task_processed = True
                # Note: do NOT break so all task failures for this record are processed.
                # ### Add manual instructions to .txt file
                # link_value = current_data.get(config['url_col'], "")
                # if next_attempt > 3:
                #     link_value = "No link yet make sure you send email and Max is CC'd"
                # rec_str = f"{record}\n" \
                #         f"{current_data.get('replay_email', '')}\n" \
                #         f"{current_data.get('email_addtl_contact', '')}\n" \
                #         f"{link_value}\n" \
                #         f"{template}\n\n"
                # task_fail_strings[f"{task}_fail"].append(rec_str)


        # ----- Process Default Case: QC Passed and Payment and (possibly) longitudinal study invite -----
        # Process default if no critical, verification, or task failure was detected.
        if not critical_handled and not task_processed and not verif_handled:
            update_df.loc[update_df['record_id'] == record, 'qc_passed'] = 1
            # Get current day of week as integer (Monday=1, Sunday=7)
            today_day_of_week = datetime.now().isoweekday()

            # Add name as employee_name column
            update_df.loc[update_df['record_id'] == record, 'employee_name'] = myname

            # Update the field for this record records in update_df
            update_df.loc[update_df['record_id'].isin([record]), 'pay_day_ofweek'] = today_day_of_week

            # If email_addtl_contact is not NaN, set send_pay_confirm to 1
            if pd.notna(current_data.get('email_addtl_contact', np.nan)):
                update_df.loc[update_df['record_id'] == record, 'send_pay_confirm'] = 1

            # Process longitudinal study interest and correct time since last sp use:
            if pd.notna(current_data.get('interested_spstudy', np.nan)) and (current_data.get('sp_dayslastuse',np.nan) > 40):
                longitudinalinvites_string += f"{current_data.get('interested_spstudy', '')}\n"
            
            # Payment processing
            row = df.loc[df['record_id'] == record]
            dates = []
            for date in ['datedone_consent_baseline',
                        'datedone_screening_survey',
                        'datedone_sms_verification',
                        'datedone_screening_result',
                        'datedone_eligibile',
                        'datedone_survey_perception_substance_use',
                        'datedone_family_history_and_asi',
                        'datedone_visual_ch_task',
                        'datedone_prl_task',
                        'datedone_auditory_ch_task',
                        'datedone_spacejunk_game',
                        'datedone_validity_checks',
                        'datedone_clarification_survey',
                        'datedone_answer_checks',
                        'datedone_extra_questions']:
                if isinstance(row[date].values[0], str):
                    dates.append(datetime.strptime(row[date].values[0], "%Y-%m-%d"))
            dates.sort(reverse=True)
            latestdate = dates[0].strftime("%m/%d/%Y")
            paymentlink = row['payment_url'].values[0]
            commments = np.nan
            if df.loc[df['record_id'] == record, 'payment_pref'].values[0] < 2:
                patype = 'Amazon-delivered physical VISA card'
                paydeets = df.loc[df['record_id'] == record, 'payment_address_bl'].values[0]
            elif (df.loc[df['record_id'] == record, 'payment_pref'].values[0] > 4) and (df.loc[df['record_id'] == record, 'payment_pref'].values[0] < 6):
                patype = 'Digital US Bank Prepaid Gift Card'
                paydeets = df.loc[df['record_id'] == record, 'payment_email_bl'].values[0]
                commments = f"{df.loc[df['record_id'] == record, 'firstname'].values[0]} {df.loc[df['record_id'] == record, 'lastname'].values[0]}"
                # --- Add row to digitalcard_template for digital gift card ---
                new_card_row = {col: np.nan for col in digitalcard_template.columns}
                new_card_row['Load Amount'] = '$40'
                new_card_row['Last Name '] = df.loc[df['record_id'] == record, 'lastname'].values[0]
                new_card_row['First Name '] = df.loc[df['record_id'] == record, 'firstname'].values[0]
                new_card_row['Email Address'] = paydeets
                new_card_row['Fulfill by Client Program'] = '1'
                new_card_row['Card Type'] = '1'
                digitalcard_template = pd.concat([digitalcard_template, pd.DataFrame([new_card_row])], ignore_index=True)
            elif df.loc[df['record_id'] == record, 'payment_pref'].values[0] > 5:
                patype = "physical VISA mailed by third party postal carrier (eg. USPS/UPS)"
                paydeets = df.loc[df['record_id'] == record, 'payment_address_bl'].values[0]
            else:
                patype = 'Amazon Gift Card'
                paydeets = df.loc[df['record_id'] == record, 'payment_email_bl'].values[0]
            
            expensesheet = pd.DataFrame({
                'Date of Participation:  (month/day/year)': [latestdate],
                'Date of Payment:          (month/day/year)': [""],
                'Name of Yale Researcher requesting payment:': [myname],
                'Name of Yale Researcher providing payment:': ["Catalina Mourgues"],
                'Amount of Payment:      (in USD)': ["$40"],
                'Subject ID #:': [record],
                'Type of Participation or Assessment Point:            **Use the same verbiage as referenced in the HIC Economic Considerations**                                                           List all activities separately, including parking, miles, food or incidentals.': ["Completion of Aim 7; baseline tasks"],
                'Type of Payment:      (Cash, Gift Card, etc.)': [patype],
                'Delivery Detail (email, address)': [paydeets],
                'Payment Instrument Link': [paymentlink],
                'Comments /Confirmation': [commments]
            })
            expensesheets.append(expensesheet)
        
        # Apply any updates queued in record_updates
        for col, val in record_updates.items():
            if col in update_df.columns:
                update_df.loc[update_df['record_id'] == record, col] = val

    # ----- After Processing All Records, Write Out Files for Each Condition -----
    # ### Used in old version when we wrote instructions to .txt files
    # def write_condition_file(condition, header, list_of_strings):
    #     if list_of_strings:
    #         master_string = header + "\n\n" + "".join(list_of_strings)
    #         file_path = f"{date_directory}{condition}.txt"
    #         with open(file_path, "w") as f:
    #             f.write(master_string)
    #         print(f"***\n{len(list_of_strings)} participants to process in {condition} (file: {file_path}).\nPlease run the 'COMPLETED RECORDS' cells below to finish processing these participants.\nMake sure you run the final cell marking that you completed these tasks once you are done!!!\n***")

    # skippedscreen_header = ("### Go to spydialer.com -> input their phone number -> select 'Name Lookup' -> "
    #                         "If it says 'VOIP' go to the payment instrument link -> Any additional notes on QC for this participant? -> ' VOIP'!!!\n"
    #                         "And then do NOT pay them!\nOtherwise, please mark them as QC=Passed and manually enter their payment information.")
    # critical_failures_header = ("### These people failed QC and will be denied payment for presumed fraud. "
    #                             "Send fraudulent_email_inconsistentanswers.emltpl to each email on this list")
    # inconsistent_sp_answers_header = ("### These people gave some inconsistent answers about their psychedelic use that need to be clarified/might indicate fraud. "
    #                                 "Send the listed email template to the email and include their specific link.")

    # write_condition_file("skippedscreen", skippedscreen_header, skippedscreen_strings)
    # write_condition_file("critical_failures", critical_failures_header, critical_failures_strings)
    # write_condition_file("inconsistent_sp_answers", inconsistent_sp_answers_header, inconsistent_sp_answers_strings)

    # for task in TASK_CONFIG.keys():
    #     condition_name = f"{task}_fail"
    #     task_header = f"### These people failed the {task}. Send the listed email template to the email and include their specific link."
    #     write_condition_file(condition_name, task_header, task_fail_strings[condition_name])
        if longitudinalinvites_string.strip():
            longi_header = ("#### Go to https://redcap.research.yale.edu/redcap_v14.5.25/Surveys/invite_participants.php?participant_list=1&pid=606 "
                            "-> \nSurvey Distribution Tools -> Participant List -> Add Participants -> Copy and paste these emails below ->  "
                            "Compose Survey Invitations ->  select email from right hand box ->  scroll to the bottom left hand drop down menu "
                            "to select a previously sent email ->  \npick any that start with 'Hello, prior Powers lab participant!' -> "
                            "Make the subject line: 'Powers Lab @ Yale Online Research: Invitation to Longitudinal Study' -> "
                            "Double check to make sure the emails are checked in the upper right hand box")
            date_directory = f"{BASE_PATH}/qc_to_dos/{datetime.now().strftime('%Y-%m-%d')}/"
            os.makedirs(date_directory, exist_ok=True)
            file_path = f"{date_directory}longitudinalinvites.txt"
            with open(file_path, "w") as f:
                f.write(longi_header + "\n\n" + longitudinalinvites_string)
            num_longi = len(longitudinalinvites_string.strip().splitlines())


    ### If there are Yale students who have finished, process them in vectorized manner. 
    if len(rec_to_check_yale)>0:
        # Students in yale qc fail list just need qc pass set to 0
        update_df.loc[update_df['record_id'].isin(yale_qc_fail), 'qc_passed'] = 0

        # Sudents not in the fail list pass and credit_yn is set to 1
        update_df.loc[~(update_df['record_id'].isin(yale_qc_fail)) & (update_df['student_yn']>0), ['qc_passed','credit_yn']] = 1

        # Students in each of the mental health groups need an email sent depending on the issue. 
        for mentalhealth_email in ["al_yale_student_mentalhealth_mdd_si", "al_yale_student_mentalhealth_psychosis", "al_yale_student_mentalhealth_substances"]:
            students = globals()[mentalhealth_email]
            if students:
                update_df.loc[update_df['record_id'].isin(students), mentalhealth_email] = 1

        # Finally, create a .txt file with a list of all the SONA IDs that need credit added and count this as one of the task flags
        with open(f"{date_directory}yale_students_sona_credit.txt", "w") as f:
            f.write("### Go to https://yale.sona-systems.com/all_exp.aspx?personal=1. Username: maxsupergreenwald\nPW: hallucinationssona!\nview uncredited timeslots -> command or control + f each SONA ID -> select each one -> Grant Credit button -> Scroll to bottom of page and hit green Process Changes button. CHECK TO MAKE SURE THE CREDIT WAS ACTUALLY GRANTED!\nALTERNATIVELY it looks like you can maybe click My Studies -> Time Slots -> Modify -> Batch Credit -> and paste all of the SONA IDs below\n")
        for sona_id in df_raw_yale['sona_id']:
            with open(f"{date_directory}yale_students_sona_credit.txt", "a") as f:
                f.write(f"{sona_id}\n")
        
        print(f"""
        *****
        THERE ARE {len(rec_to_check_yale)} YALE STUDENTS THAT NEED SONA ID CREDIT TO BE ASSIGNED! 
        Upload the SONA IDs in {date_directory}yale_students_sona_credit.txt and follow the directions :)
        <a href="https://yale.sona-systems.com/all_exp.aspx?personal=1" target="_blank">Add SONA IDs here!</a>
        *****
              """)


    # ----- Check to make sure there are no records in records_to_check that are not getting handled -----
    handled_records = set(update_df['record_id'].tolist())
    # Collect all records that were processed in any way
    all_flagged = set()
    all_flagged.update(master_qc_fail)
    all_flagged.update(yale_qc_fail)
    all_flagged.update(skippedscreen)
    all_flagged.update([rec for sublist in task_fail_strings.values() for rec in [int(s.split('\n')[0]) for s in sublist]])
    all_flagged.update([int(s.split('\n')[0]) for s in critical_failures_strings])
    all_flagged.update([int(s.split('\n')[0]) for s in inconsistent_sp_answers_strings])
    all_flagged.update([int(s.split('\n')[0]) for s in skippedscreen_strings])

    # Find records that were not handled
    unhandled_records = [rec for rec in records_to_check if rec not in handled_records and rec not in all_flagged]

    if unhandled_records:
        print(f"WARNING: The following records in records_to_check were not handled by the QC loop: {unhandled_records}")
        print("Please investigate why these records were not processed.")
    
    # ----- If a subject finished the addon questions then we just pay them regardless cuz we trust them -- add their info to the growing expensesheets and update 'addons_completed' and 'addons_payment' both to be '1'  -----
    if len(addon_finishes) > 0:
        for record in addon_finishes:
            # Add a row to the expensesheets for each record that finished the addon questions
            row = df.loc[df['record_id'] == record]
            paymentlink = row['payment_url'].values[0]
            commments = np.nan
            if df.loc[df['record_id'] == record, 'new_digital_card'].values[0] < 2:
                patype = 'Digital US Bank Prepaid Gift Card'
                paydeets = df.loc[df['record_id'] == record, 'payment_email_reconsent'].values[0]
                commments = f"{df.loc[df['record_id'] == record, 'firstname_reconsent'].values[0]} {df.loc[df['record_id'] == record, 'lastname_reconsent'].values[0]}"
                # --- Add row to digitalcard_template for digital gift card ---
                new_card_row = {col: np.nan for col in digitalcard_template.columns}
                new_card_row['Load Amount'] = '$5'
                new_card_row['Last Name '] = df.loc[df['record_id'] == record, 'lastname_reconsent'].values[0]
                new_card_row['First Name '] = df.loc[df['record_id'] == record, 'firstname_reconsent'].values[0]
                new_card_row['Email Address'] = paydeets
                new_card_row['Fulfill by Client Program'] = '1'
                new_card_row['Card Type'] = '1'
                digitalcard_template = pd.concat([digitalcard_template, pd.DataFrame([new_card_row])], ignore_index=True)
            else:
                patype = 'Amazon Gift Card'
                paydeets = df.loc[df['record_id'] == record, 'payment_email_reconsent'].values[0]
            expensesheet = pd.DataFrame({
                'Date of Participation:  (month/day/year)': row["datedone_add_on_questions"],
                'Date of Payment:          (month/day/year)': [""],
                'Name of Yale Researcher requesting payment:': [myname],
                'Name of Yale Researcher providing payment:': ["Catalina Mourgues"],#["Catalina Mourgues"],
                'Amount of Payment:      (in USD)': ["$5"],
                'Subject ID #:': [record],
                'Type of Participation or Assessment Point:            **Use the same verbiage as referenced in the HIC Economic Considerations**                                                           List all activities separately, including parking, miles, food or incidentals.': ["Completion of Aim 7; Additional Clarification Questions"],
                'Type of Payment:      (Cash, Gift Card, etc.)': [patype],
                'Delivery Detail (email, address)': [paydeets],
                'Payment Instrument Link': [paymentlink],
                'Comments /Confirmation': [commments]
            })
            expensesheets.append(expensesheet)

            # Update the record in df to mark addons completed and payment made
            update_df.loc[update_df['record_id'].isin([record]), 'addons_completed'] = 1
            update_df.loc[update_df['record_id'].isin([record]), 'addons_payment'] = 1

    # ----- Final Exports and Messages (Expense sheet, update_df, etc.) -----
    update_df.reset_index(drop=True, inplace=True)
    jsons_df = pd.concat([jsons_df, json_df], ignore_index=True)
    jsons_df.to_csv(json_path, index=False)

    # --- Save updated digitalcard_template if any new rows were added ---
    if len(digitalcard_template) > 0:
        digitalcard_template.to_csv(f"{date_directory}digital_card_form{datetime.now().strftime('%Y-%m-%d')}.csv", index=False)
        print(f"*****\nDont forget! {len(digitalcard_template)} participants also need their rows added to the 'Digital Card Form' sheet of Sharepoint -- you can find it at {date_directory}digital_card_form{datetime.now().strftime('%Y-%m-%d')}.csv\n*****")

    # --- Export expensesheet to date_directory if there is one ---
    if len(expensesheets) > 1:
        expensesheetfin = pd.concat(expensesheets)
        # # --- Check for missing Payment Instrument Link and fill from df if possible ---
        nan_payments = expensesheetfin[expensesheetfin['Payment Instrument Link'].isna()]
        if len(nan_payments) > 0:
            for index,row in nan_payments.iterrows():
                ### For each row, print "consent_baseline_url" and ask the user to input the payment_url by going to the payment instrument and pasting the url -- 
                record_id = row['Subject ID #:']
                if pd.notna(df[df["record_id"] == record_id]["pay_url"].values[0]):
                    payment_url = df.loc[df['record_id'] == record_id, 'pay_url'].values[0]
                    if pd.notna(payment_url):
                        expensesheetfin.loc[expensesheetfin['Subject ID #:'] == record_id, 'Payment Instrument Link'] = payment_url
                    else:
                        consent_baseline_url = df.loc[df['record_id'] == record_id, 'consent_baseline_url'].values[0]
                        print(f"Please go to {consent_baseline_url} and input the payment URL for record {record_id}:")
                        payment_url = input(f"Please go to {consent_baseline_url} and input the payment URL for record {record_id}: ")
                    
                if payment_url:
                    expensesheetfin.loc[expensesheetfin['Subject ID #:'].isin([record_id]), 'Payment Instrument Link'] = payment_url
        # if 'Payment Instrument Link' in expensesheetfin.columns:
        #     mask_na = expensesheetfin['Payment Instrument Link'].isna()
        #     if mask_na.any():
        #         for idx in expensesheetfin[mask_na].index:
        #             record_id = expensesheetfin.loc[idx, 'Subject ID #:']
        #             # Try to get payment_url from df
        #             payment_url = df.loc[df['record_id'] == record_id, 'payment_url'].values
        #             if len(payment_url) > 0 and pd.notna(payment_url[0]):
        #                 expensesheetfin.at[idx, 'Payment Instrument Link'] = payment_url[0]
        #         # Warn if any still NA
        #         if expensesheetfin['Payment Instrument Link'].isna().any():
        #             print("Warning: Some Payment Instrument Link values are still missing after attempting to fill from df.")
        expensesheet_file = f"{date_directory}expensesheet_{datetime.now().strftime('%Y-%m-%d')}.csv"
        expensesheetfin.to_csv(expensesheet_file, index=False)
        print(f"*****\nExported expensesheet with {len(expensesheetfin)} participants ready for upload at {expensesheet_file}\n*****")
        display(HTML(f'<a href="https://yaleedu-my.sharepoint.com/:x:/r/personal/silmilly_toribio_yale_edu/_layouts/15/Doc.aspx?sourcedoc=%7B989CA093-98FB-412F-A0BA-F99D26B72AAF%7D&file=Participant%20Payment%20Request%20Psychedelics%20MSG.xlsx&action=default&mobileredirect=true&DefaultItemOpen=1" target="_blank">Upload Expense Sheet Here</a>'))
    elif len(expensesheets) > 0:
        expensesheetfin = expensesheets[0]
        # --- Check for missing Payment Instrument Link and fill from df if possible ---
        nan_payments = expensesheetfin[expensesheetfin['Payment Instrument Link'].isna()]
        if len(nan_payments) > 0:
            for index,row in nan_payments.iterrows():
                ### For each row, print "consent_baseline_url" and ask the user to input the payment_url by going to the payment instrument and pasting the url -- 
                record_id = row['Subject ID #:']
                if df[df["record_id"] == record_id]["pay_url"].notna().any():
                    payment_url = df.loc[df['record_id'] == record_id, 'pay_url'].values[0]
                    if pd.notna(payment_url):
                        expensesheetfin.loc[expensesheetfin['Subject ID #:'] == record_id, 'Payment Instrument Link'] = payment_url
                    else:
                        consent_baseline_url = df.loc[df['record_id'] == record_id, 'consent_baseline_url'].values[0]
                        print(f"Please go to {consent_baseline_url} and input the payment URL for record {record_id}:")
                        payment_url = input(f"Please go to {consent_baseline_url} and input the payment URL for record {record_id}: ")
                    
                if payment_url:
                    expensesheetfin.loc[expensesheetfin['Subject ID #:'].isin([record_id]), 'Payment Instrument Link'] = payment_url

        expensesheet_file = f"{date_directory}expensesheet_{datetime.now().strftime('%Y-%m-%d')}.csv"
        expensesheetfin.to_csv(expensesheet_file, index=False)
        print(f"*****\nExported expensesheet for one participant ready for upload at {expensesheet_file}\n*****")
        display(HTML(f'<a href="https://yaleedu-my.sharepoint.com/:x:/r/personal/silmilly_toribio_yale_edu/_layouts/15/Doc.aspx?sourcedoc=%7B989CA093-98FB-412F-A0BA-F99D26B72AAF%7D&file=Participant%20Payment%20Request%20Psychedelics%20MSG.xlsx&action=default&mobileredirect=true&DefaultItemOpen=1" target="_blank">Upload Expense Sheet Here</a>'))
    else:
        print("Nobody gettin paid today")

    if len(update_df) > 0:
        print(f"***\n{len(update_df)} records to modify in REDCap today -- please run 'COMPLETED RECORDS' Cell below!!\n***")
    else:
        print(f"***\nNo records to modify today!\n***")
    
    if (len(expensesheets)>0) and (longitudinalinvites_string != ""): 
        print(f"""*****\nThere are {num_longi} participants that need to be added to the longitudinal study! Go to {date_directory}longitudinalinvites.txt and follow the directions :)\n<a href="https://redcap.research.yale.edu/redcap_v14.5.25/Surveys/invite_participants.php?participant_list=1&pid=606" target="_blank">LINK TO LONGITUDINAL STUDY REDCAP*****""")

# update flag so that people can run the later cells
qc_run = True

################################################################################################################
############## Human Error handling -- Set flags to track whether users have completed all tasks ##############
################################################################################################################
#### Generate flag files based on records_to_screen and records_to_check
def generate_flag_files(this_qc_path, records_to_screen, records_to_check, addon_finishes, expensesheets):
    ### Screening flags
    if len(records_to_screen) > 0:
        # with open(os.path.join(this_qc_path, "TASKS_SCREENS_INCOMPLETE.txt"), "w") as f:
        #     f.write(myname)
        with open(os.path.join(this_qc_path, "REDCAP_SCREENS_INCOMPLETE.txt"), "w") as f:
            f.write(myname)
    else:
        # with open(os.path.join(this_qc_path, "TASKS_SCREENS_NA.txt"), "w") as f:
        #     f.write(myname)
        with open(os.path.join(this_qc_path, "REDCAP_SCREENS_NA.txt"), "w") as f:
            f.write(myname)

    ### Full records flags
    if len(records_to_check) > 0 or len(addon_finishes) > 0 or (len(rec_to_check_yale) > 0):
        # if there are longitudinal invites needed, then there are tasks to do 

        if (longitudinalinvites_string.strip()) or (len(rec_to_check_yale) > 0):
            with open(os.path.join(this_qc_path, "TASKS_FULLRECORDS_INCOMPLETE.txt"), "w") as f:
                f.write(myname)
        else:
            with open(os.path.join(this_qc_path, "TASKS_FULLRECORDS_NA.txt"), "w") as f:
                f.write(myname)

        with open(os.path.join(this_qc_path, "REDCAP_FULLRECORDS_INCOMPLETE.txt"), "w") as f:
            f.write(myname)
        if len(expensesheets) > 0:
            with open(os.path.join(this_qc_path, "PAYMENTS_FULLRECORDS_INCOMPLETE.txt"), "w") as f:
                f.write(myname)
        else:
            with open(os.path.join(this_qc_path, "PAYMENTS_FULLRECORDS_NA.txt"), "w") as f:
                f.write(myname)
    else:
        with open(os.path.join(this_qc_path, "TASKS_FULLRECORDS_NA.txt"), "w") as f:
            f.write(myname)
        with open(os.path.join(this_qc_path, "REDCAP_FULLRECORDS_NA.txt"), "w") as f:
            f.write(myname)
        with open(os.path.join(this_qc_path, "PAYMENTS_FULLRECORDS_NA.txt"), "w") as f:
            f.write(myname)

generate_flag_files(date_directory, records_to_screen, rec_to_check_reg, addon_finishes, expensesheets)

#########################################
##### Functions for UPDATING the flags ##
#########################################
# Update flag files for REDCap updates
def update_redcap_flag(base_path, flag_name):
    flag_path = os.path.join(base_path, flag_name + ".txt")
    if os.path.exists(flag_path):
        os.remove(flag_path)
        with open(flag_path.replace("INCOMPLETE.txt", "COMPLETE.txt"), "w") as f:
            f.write(myname)
        print(f"{flag_name} updated to COMPLETE.")
    else:
        print(f"No {flag_name} found -- no actions were required today.")
        
# Update flag files for task/payment completion where we basically just aas ppl 
def update_task_flag(base_path, flag_name, prompt_message, thingtocomplete):
    flag_path = os.path.join(base_path, flag_name + ".txt")
    if os.path.exists(flag_path):
        user_input = input(prompt_message)
        if user_input.lower() == "yes":
            os.remove(flag_path)
            with open(flag_path.replace("INCOMPLETE.txt", "COMPLETE.txt"), "w") as f:
                f.write(myname)
            print(f"{flag_name} updated to COMPLETE.")
        else:
            print(f"Please complete {thingtocomplete} and run this cell again.")
    else:
        print(f"No {flag_name} found to update.")


***
1 Screening records to check the numbers for and at least 0 who need to be rejected!
Please run all cells in order going down from NEW RECORDS SCREENING
****


In [4]:
#### WHAT HAPPENED FOR EACH RECORD IN QC -- DEBUGGING
master_qc_lists = [zero, failedAttnCheck, negative, zero_vch, negative_vch, 
failed_sp_qc, failed_micro_qc, repeat_nomic_q, meant_micro, no_lose_stay, failed_usetime_qc, non_responders, worse_than_chance, 
nosurvey1, no_sms_verification, illogical_year, illogical_life, illogical_6month, no_prl, no_ach, no_vch, failed_new_qc,
nanresponses_screen,nanresponses,wrong_recent,ineligibile_yalies]

master_qc_names = ["zero","failedAttnCheck","negative","zero_vch","negative_vch",
                   "failed_sp_qc","failed_micro_qc","repeat_nomic_q","meant_micro","no_lose_stay","failed_usetime_qc","non_responders","worse_than_chance",
                   "nosurvey1","no_sms_verification","illogical_year","illogical_life","illogical_6month","no_prl","no_ach","no_vch","failed_new_qc",
                   "nanresponses_screen","nanresponses","wrong_recent","ineligibile_yalies"]

recordresults = {}
for record in update_df["record_id"]:
    for lister, name in zip(master_qc_lists, master_qc_names):
        if record in lister:
            if record not in recordresults:
                recordresults[record] = []
            recordresults[record].append(name)
    if record not in recordresults:
        print(f"{record} not in any QC lists and should be paid and/or have credit given!")
    else:
        print(f"Record {record} in: {recordresults[record]}")


# Debug snippet to trace why records were added to DontPassGo
def debug_dontpassgo_reasons(records_to_check, DontPassGo, df, df_ips, ineligibles, ineligibileemails, trueips, ineligibile_ips):
    """
    Debug function to show why each record in DontPassGo was flagged
    """
    print("=== DEBUGGING DontPassGo RECORDS ===\n")
    
    for record in DontPassGo:
        print(f"Record {record} was flagged for:")
        reasons = []
        
        # Check forbidden orgs/countries
        if record in records_to_check:
            record_ips = df_ips[df_ips['record_id'] == record]
            if not record_ips.empty:
                forbidden_orgs = ["AS14061 DigitalOcean, LLC","AS396356 Latitude.sh",
                                 "AS5650 Frontier Communications of America, Inc.",
                                 "AS25769 Garden Valley Telephone Company","AS21859 Zenlayer Inc",
                                  "AS22200 Bloomingdale Communications Inc.",
                                 "AS8167 V tal", "AS11878 tzulo, inc.", #"AS63023 GTHost",
                                 "AS393398 1515 ROUNDTABLE DR PROPERTY, LLC",
                                 'AS396956 Meriwether Lewis Electric Cooperative', 'AS9009 M247 Europe SRL', 'AS16276 OVH SAS', 
                                 'AS136787 PacketHub S.A.', 'AS62240 Clouvider', 'AS60068 Datacamp Limited',
                                 'AS6079 RCN', 'AS13285 TalkTalk Communications Limited',
                                 'AS6461 Zayo Bandwidth', 'AS174 Cogent Communications', 'AS212238 Datacamp Limited', 
                                 'AS6300 Consolidated Communications, Inc.']
                forbidden_country_names = ["Nigeria","Ghana"]
                
                if record_ips['org'].isin(forbidden_orgs).any():
                    reasons.append(f"  - Forbidden organization: {record_ips['org'].iloc[0]}")
                if record_ips['country_name'].isin(forbidden_country_names).any():
                    reasons.append(f"  - Forbidden country: {record_ips['country_name'].iloc[0]}")
        
        # Check challenge questions
        thisrecdf = df[df['record_id'] == record]
        if not thisrecdf.empty:
            thisrec = thisrecdf.iloc[0]
            if (thisrec.get("control_group", 0) > 0) and (thisrec.get('psycheduse_yn', 0) > 1):
                reasons.append("  - Control group participant claiming SP use")
            
            if (thisrec.get('student_yn', 0) > 0) and (thisrec.get('howtheyfoundus', 0) < 2):
                reasons.append("  - Student claiming to have found study through Yale intro psych")
        
        # Check email duplicates
        if record in records_to_check:
            thisrecdf = df[df['record_id'] == record]
            if not thisrecdf.empty:
                thisguysemails = (thisrecdf['payment_email_bl'].tolist() + 
                                thisrecdf['email_addtl_contact'].tolist() + 
                                thisrecdf['interested_cntrlstudy'].tolist() + 
                                thisrecdf["interested_spstudy"].tolist())
                
                for email in thisguysemails:
                    if pd.notna(email) and email in ineligibileemails:
                        # Find which ineligible record has this email
                        matching_ineligible = ineligibles[
                            (ineligibles['payment_email_bl'] == email) |
                            (ineligibles['email_addtl_contact'] == email) |
                            (ineligibles['interested_cntrlstudy'] == email) |
                            (ineligibles['interested_spstudy'] == email)
                        ]
                        if record not in matching_ineligible['record_id'].tolist():
                            reasons.append(f"  - Email duplicate: {email} (matches records: {matching_ineligible['record_id'].tolist()})")
        
        # If no reasons found, check if they were added elsewhere
        if not reasons:
            reasons.append("  - Added in earlier processing (check forbidden_orgs, forbidden_country_names, or challenge questions)")
        
        for reason in reasons:
            print(reason)
        print()

# Run the debug function
if 'DontPassGo' in globals() and len(DontPassGo) > 0:
    debug_dontpassgo_reasons(records_to_screen, DontPassGo, df, df_ips, ineligibles, ineligibileemails, trueips, ineligibile_ips)
else:
    print("No records in DontPassGo to debug")
   

No records in DontPassGo to debug


<span style="font-size: 80px;"> NEW RECORDS SCREENING </span>

## Step 1: Open [SpyDialer](https://www.spydialer.com/) and [IPQuality Score](https://www.ipqualityscore.com/user/dashboard) then run this cell
### You will be prompted to enter 'y' or 'n' for every record's phone_number and IP address to indicate whether the phone is a VOIP or the IP address has bot or abuse activity with a fraud score higher than 80 ('y') or not ('n').
### Go to [IPQuality Score](https://www.ipqualityscore.com/user/phone-number-validation-api/lookup/number) and enter the phone number (remove the US/Canada default location options) if SpyDialer says it can't take the number cuz it's outside its database! If it says "False" for both "Recent Abuse" and "VOIP" but "True" for valid, then we can mark the record 'n'. If it doesn't work, just enter "?" for that record and Max will handle it. 
username = maximillian.greenwald@yale.edu

pw = kwDgTgWSsqD8#!@


In [5]:
# Initialize lists if they don't exist yet
pass_go = []
email_max = []

if len(records_to_screen)>0:
    if qc_run == True:
        ### Have user input whether or not each number that needs to be checked is a VOIP or not! 
        if len(spydialer_check) > 0:
            for record in spydialer_check:        
                phone = int(df.loc[df['record_id']==record,'phone_number'].values[0])
                ip = df_ips.loc[df_ips['record_id']==record,'ip'].values[0]
                print(f"record:{record}")
                print(f"Phone:\n{phone}")
                print(f"IP Address:\n{ip}")
                verdict = input(f"Is {record}'s phone number ({phone}) a VOIP number (you can copy it from beneath the cell)? (y/n). And Is their IP address Enter 'y' if it IS a VOIP and 'n' if it is NOT! If spydialer doesn't work try IPQuality score (see above).  If not sure or something else about the number is suspicious, enter '?', And if you ran this cell in error enter 'stop' to start over")
                
                if verdict.lower() == 'y':
                    DontPassGo.append(record)
                    print(f"Marking {record} as VOIP and to be marked as fraud! If this is an error, please re-run this cell and enter 'n' for {record} instead.")
                elif verdict.lower() == 'n':
                    pass_go.append(record)
                    print(f"Marking {record} as good :)")
                elif verdict.lower() == '?':
                    email_max.append(record)
                    print(f"Marking {record} as needing to be emailed to Max for further review! If this is an error, please re-run this cell and enter 'y' or 'n' for {record} instead.")
                elif verdict.lower() == 'stop':
                    print("Stopping the verification process")
                    pass_go = None
                    DontPassGo = None
                    email_max = None
                    break
                else:
                    print(f"Invalid input '{verdict}'. Please enter 'y', 'n', '?' or 'stop'")
                    # Re-ask for this record and attempt to assign it again
                    verdict = input(f"Invalid input '{verdict}'. Please enter 'y', 'n', '?' or 'stop. I repeat -- is {phone} a VOIP number? (y/n). '?' if not sure and 'stop' to start over ")
                    if verdict.lower() == 'y':
                        DontPassGo.append(record)
                    # If they aren't a VOIP, check to make sure they dont have a repeat IP address -- if they do, they belong only on the sus_ips list
                    elif verdict.lower() == 'n':
                        if record in sus_ips:
                            continue
                        else:
                            pass_go.append(record)
                    elif verdict.lower() == '?':
                        email_max.append(record)
                    elif verdict.lower() == 'stop':
                        print("Stopping the verification process")
                        pass_go = None
                        DontPassGo = None
                        email_max = None
                        break
                    else:
                        print(f"Invalid input again '{verdict}'. Stopping verication process")
                        break

            # Refine sus_ips to not include folks who downright fail d/t DontPassGo
            DontPassGo = [int(x) for x in DontPassGo]
            sus_ips = [x for x in sus_ips if x not in DontPassGo]

            # Let them know they can move to next cell
            print(f"All done! You are good to move onto the next cell :)")
        
        # If no records to run, then just let em know
        else:
            print("No records requiring phone number checking in spydialer -- you're good to move onto the next cell :)")

        # So long as they did QC and run this cell, then we let them move on    
        spydialed = True
        print(f"All done -- go ahead and update REDCap with the cell below :)")

    else:
        print("Please run above QC cell before seeing who needs to be checked on spydialer ")
else: 
    print("No records to screen today -- you're good to move onto the COMPLETED RECORDS section :)")
    spydialed = True

record:2750
Phone:
19014964642
IP Address:
162.224.6.235
Marking 2750 as good :)
All done! You are good to move onto the next cell :)
All done -- go ahead and update REDCap with the cell below :)


In [6]:
### Now that we are focusing on the longitudinal study, check to make sure that everyone is onboard to do the main study
# Initialize lists for records that don't meet criteria
sp_lastuse_less_than_42 = []
interested_spstudy_missing = []

# Loop through records_to_screen
for record_id in records_to_screen:
    # Get the record data
    record_data = df[df['record_id'] == record_id]
    
    if len(record_data) == 0:
        continue
    
    # Check sp_lastuse_days_screen >= 42
    sp_lastuse = record_data['sp_lastuse_days_screen'].values[0]
    if pd.isna(sp_lastuse) or sp_lastuse < 42:
        sp_lastuse_less_than_42.append(record_id)
    
    # Check interested_spstudy is not na
    interested = record_data['interested_spstudy'].values[0]
    if pd.isna(interested):
        interested_spstudy_missing.append(record_id)

print(f"Records with sp_lastuse_days_screen < 42 or missing: {len(sp_lastuse_less_than_42)}")
print(f"Record IDs: {sp_lastuse_less_than_42}")
print(f"\nRecords with interested_spstudy missing: {len(interested_spstudy_missing)}")
print(f"Record IDs: {interested_spstudy_missing}")




Records with sp_lastuse_days_screen < 42 or missing: 0
Record IDs: []

Records with interested_spstudy missing: 0
Record IDs: []


In [ ]:
### Remove folks who do not qualify for longitudinal study 
for rec in pass_go:
    if rec in sp_lastuse_less_than_42 or rec in interested_spstudy_missing:
        pass_go.remove(rec)

## 2: *After Step 1* -- Update REDCap so they can move on/be failed!

In [7]:
### First check to mamke sure they actually did the spydialer check
### If they have reviewed the numbers/ip addresses, then update screening_pass field to indicate whether or not they are allowed to move on! (also mark as QC failed if they fail screening)
if (spydialed == True):
    redcapcheck = input("Are you sure you want to update REDCap with the screening pass/fail? If so, type 'yes'. Type anything else to exit")
    if redcapcheck.lower() == 'yes':
        if (len(DontPassGo) >0) or (len(pass_go) > 0) or (len(cannabis_screens) > 0):

            # Create DF to push to REDCap
            screenupdates = df_screens.copy()
            screenupdates = screenupdates[screenupdates['record_id'].isin(records_to_screen)]

            ### People who fail
            screenupdates.loc[screenupdates['record_id'].isin(DontPassGo),['screening_pass','qc_passed']] = 0
            screenupdates.loc[(screenupdates['record_id'].isin(DontPassGo)) & (screenupdates['submit_screen_v3'] < 2),'ineligibile_fraud'] = 1

            
            ### People who pass 
            screenupdates.loc[df['record_id'].isin(pass_go),'screening_pass'] = 1
            # Normal records not doing cannabis or longitudinal screens who are kosher and request an email must get one
            screenupdates.loc[(screenupdates['record_id'].isin(pass_go)) & 
                 (~screenupdates['record_id'].isin(cannabis_screens)) & 
                 (~screenupdates['record_id'].isin(longitudinal_screens)) & 
                 (screenupdates['submit_screen_v3'] < 2).values, 'eligible_notify'] = 1
            
            ### sus IPS who indicate they r cool with being emailed
            screenupdates.loc[(screenupdates['record_id'].isin(sus_ips)) & 
                 (screenupdates['submit_screen_v3'] < 2).values, 'ip_zoom_invite'] = 1
            
            ### Cannabis peeps who pass
            screenupdates.loc[(screenupdates['record_id'].isin(cannabis_screens)) & (df['record_id'].isin(pass_go)),['screening_pass','zoom_eligibility','cb_email_yn']] = 1
            
            ### Longitudinal screens who pass
            screenupdates.loc[(screenupdates['record_id'].isin(longitudinal_screens)) & (screenupdates['record_id'].isin(pass_go)),['waiting_emailed_yn','longitudinal_waiting_confirmation_better','longitudinal_continue_link']] = 1

            ### People who had weird phone numbers
            screenupdates.loc[screenupdates['record_id'].isin(email_max), 'email_max'] = 1

            screen_alert_fields = ['screening_pass','qc_passed','cb_email_yn',"waiting_emailed_yn","ineligibile_fraud","eligible_notify","ip_zoom_invite",'longitudinal_waiting_confirmation_better','longitudinal_continue_link','zoom_eligibility','max_number_followup']
            screenupdates[screen_alert_fields] = screenupdates[screen_alert_fields].astype('Int64')
            screenupdates = screenupdates[screen_alert_fields + ['record_id'] + ["ineligibilty_reason"]]
            project.import_records(to_import=screenupdates, import_format='df')
            update_redcap_flag(date_directory, "REDCAP_SCREENS_INCOMPLETE")

            print(f"Screening records updated in REDCap! You are all done with Screening")
            if len(update_df) > 0:
                print("Just make sure you run the 'COMPLETED RECORDS' cell below to finish processing these participants :)")
        else:
            print("No new screening records requiring updates; REDCap NOT updated! You're good to move onto the next cell :)")
        
        screens_updated = True

    else:
        print("That's OK! Come back when you're ready, buddy!")
else:
    print("Please run above cell and fill out who has a VOIP or not before we update REDCap")


REDCAP_SCREENS_INCOMPLETE updated to COMPLETE.
Screening records updated in REDCap! You are all done with Screening


<span style="font-size: 80px;">COMPLETED RECORDS</span>

## 1. Update REDCap's tasks, QC field, etc.

In [ ]:
### Update REDCap only if we've run the QC cell.
### Main processing script: 
if (qc_run == True):
    yesnoupdate = input("Are you sure you want to update REDCap with the full record updates? If so, type 'yes'. Type anything else to exit")
    if yesnoupdate.lower() == 'yes':
        if len(update_df)>0:
            # Convert float columns that can be integers to integers cuz that's what redcap likes. 
            for col in updatecols:
                if len(update_df)>1:
                    if pd.api.types.is_float_dtype(update_df[col]):
                            # Check if all non-null values are whole numbers
                            if (update_df[col].dropna() % 1 == 0).all():
                                update_df[col] = update_df[col].astype("Int64") #''
                            else:
                                print(f"Column {col} contains non-integer floats - leaving as float")
                elif len(update_df)==1:
                    if (pd.api.types.is_float_dtype(update_df[col])) and (update_df[col].isna().sum()<1):
                        # Check if all non-null values are whole numbers
                        if (update_df[col].iloc[0] % 1 == 0):
                            # print(f"updated {col}")
                            update_df[col] = update_df[col].astype("Int64") #'Int64'
                        # else:
                            # print(f"Column {col} contains non-integer floats - leaving as float")
                else:
                    print(f"There are no records to update?? ")
                            

            update_df = update_df[updatecols + ['record_id']]

            # And remove the "index" column
            if "index" in update_df.columns:
                update_df = update_df.drop(columns=["index"])

            # Befor ever importing records with an overwrite, save a backup of the dataframe with today's date
            df_og.to_csv(f"{date_directory}df_og_{datetime.now().strftime('%Y-%m-%d')}.csv",index=False)

            ### Send it back to redcap!
            project.import_records(to_import=update_df, import_format='df',overwrite='overwrite')

            print(f"***\n{len(update_df)} records updated in REDCap. You're fine to move to the next cell :)\n***")

        else:
            print("0 records requiring update detected -- REDCap not updated! You're fine to move to the next cell :)")
        
        update_redcap_flag(date_directory, "REDCAP_FULLRECORDS_INCOMPLETE")
        recs_updated = True
        
    else:
        print("That's OK! Come back when you're ready, buddy!")

else:
    print("Exiting WITHOUT updating REDCap!")

## 2. Run this cell once you have sent out all the appropriate emails and such

In [ ]:
### Update the full records flag in date_directory to indicate that all of today's tasks were completed!
if (recs_updated == True):
    update_task_flag(date_directory, "TASKS_FULLRECORDS_INCOMPLETE", "Have you sent all of the emails you needed to send, Added SONA credit, AND updated REDCap? Type 'yes' to confirm -- anything else to abort", f"all the tasks at {date_directory}")
else:
    print("You should only run this cell after you do all the above steps -- update REDCap and grun the main QC script first! (NOTE it's OK to run even if there are not REDCap updates -- it should print 'no updates')")


# 3. Run this cell if there are payments AFTER you have successfully [uploaded the payments to sharepoint](https://yaleedu-my.sharepoint.com/:x:/r/personal/silmilly_toribio_yale_edu/_layouts/15/Doc.aspx?sourcedoc=%7B989CA093-98FB-412F-A0BA-F99D26B72AAF%7D&file=Participant%20Payment%20Request%20Psychedelics%20MSG.xlsx&action=default&mobileredirect=true&DefaultItemOpen=1) and emailed Cata

In [ ]:
### Update the full records flag in date_directory to indicate that all payments from today were uploaded!
# Update flag files for payments
if (recs_updated == True):
    update_task_flag(date_directory, "PAYMENTS_FULLRECORDS_INCOMPLETE", "Have you uploaded payments to SharePoint and emailed Silmilly? Type 'yes' to confirm. Anything else to exit ","participant payments")
else:
    print("You should only run this cell after you do all the above steps -- update REDCap and run the main QC script first!")

Looking at new, bad IPs:

In [ ]:
bad_ips =[
    "172.56.108.128",
    "37.19.220.13",
    "102.129.234.159",
    "71.57.13.203",
    "212.32.49.42",
    "209.34.30.135",
    "172.56.46.222",
    "100.12.70.52",
    "172.59.193.21"
    
    
    
]

df_ips = pd.read_csv(ip_path)


df_ips_new_assholes = df_ips[df_ips['ip'].isin(bad_ips)]
orgs_to_ban = df_ips_new_assholes['org'].unique().tolist()
print(orgs_to_ban)
df_ips_new_assholes[['record_id','ip','org']]

Are people hacking our ssytem to send bad data?

In [ ]:
# for index,row in df.iterrows()['task_data_prltask']:
#     if 'H4sIAAAAAA' not in x:
for task in ['task_data_ach_task_short_baseline','task_data_prltask','task_data_vch_short_psychedelic_bl','task_data_spacejunk_bl']:
    fake_task_mask = df[task].apply(lambda x: isinstance(x, str) and 'H4sIAAAAAA' not in x)
    if fake_task_mask.any():
        print(f"WARNING: The following records have what looks like fake data for {task}: {df[fake_task_mask][['record_id', task]]}")
    else:
        print(f"No fake data detected for {task}")

In [ ]:
# print(len(df_ips[df_ips['org'].isin(['AS21928 T-Mobile USA, Inc.', 'AS701 Verizon Business','AS7922 Comcast Cable Communications, LLC'])]))
['AS6461 Zayo Bandwidth', 'AS174 Cogent Communications', 'AS212238 Datacamp Limited', 'AS6300 Consolidated Communications, Inc.']
#'AS7922 Comcast Cable Communications, LLC', 'AS21928 T-Mobile USA, Inc.', 'AS701 Verizon Business', 

In [ ]:
# ### Why is the current record ending up in ineligibile_yalies? Make each .extend into an explicit list and then check if the record is in that list
# tooold = df_raw_yale.loc[df_raw_yale['age_v2']>65,'record_id'].tolist()
# tooyoung = df_raw_yale.loc[df_raw_yale['age_v2']<18,'record_id'].tolist()
# badcog = df_raw_yale.loc[df_raw_yale['cognition_screener_v2']>0,'record_id'].tolist()
# seizures = df_raw_yale.loc[df_raw_yale['seizure_hx_v2']>0,'record_id'].tolist()
# intox = df_raw_yale.loc[df_raw_yale['intox_screen_v2']>0,'record_id'].tolist()
# lowraven = df_raw_yale.loc[df_raw_yale['raven_total_score_v2']<1,'record_id'].tolist()
# highfreq = df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<28 & (df_raw_yale['cannabis_frequency']>9),'record_id'].tolist()
# modfreq = df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<14 & (df_raw_yale['cannabis_frequency'].isin([8,9])),'record_id'].tolist()
# lowfreq = df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<7 & (df_raw_yale['cannabis_frequency']==7),'record_id'].tolist()
# verylowfreq = df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<3,'record_id'].tolist()
# nocomputer = df_raw_yale.loc[df_raw_yale['no_computer']>0,'record_id'].tolist()

# for record in rec_to_check_yale:
#     for condition, lst in zip(["tooold","tooyoung","badcog","seizures","intox","lowraven","highfreq","modfreq","lowfreq","verylowfreq","nocomputer"],
#                               [tooold,tooyoung,badcog,seizures,intox,lowraven,highfreq,modfreq,lowfreq,verylowfreq,nocomputer]):
#         if record in lst:
#             print(condition)
# # if len(rec_to_check_yale)>0:
# #     ineligibile_yalies = []
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['age_v2']>65,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['age_v2']<18,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['cognition_screener_v2']>0,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['seizure_hx_v2']>0,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['intox_screen_v2']>0,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['raven_total_score_v2']<1,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<28 & (df_raw_yale['cannabis_frequency']>9),'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<14 & (df_raw_yale['cannabis_frequency'].isin([8,9])),'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<7 & (df_raw_yale['cannabis_frequency']==7),'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['activecannabisuse_lastuse']<3,'record_id'].tolist())
# #     ineligibile_yalies.extend(df_raw_yale.loc[df_raw_yale['no_computer']>0,'record_id'].tolist())